1. Configuración Inicial y Metadatos

In [20]:

"""
ANÁLISIS HÍBRIDO DE AUSENTISMO LABORAL - EQUIPO 15
Fecha: 25/09/2025
Versión: 2.0
Autor: Equipo de Análisis de Datos - Oriac Gimeno
Descripción: Análisis completo de factores que influyen en el ausentismo laboral
"""

'\nANÁLISIS HÍBRIDO DE AUSENTISMO LABORAL - EQUIPO 15\nFecha: 25/09/2025\nVersión: 2.0\nAutor: Equipo de Análisis de Datos - Oriac Gimeno\nDescripción: Análisis completo de factores que influyen en el ausentismo laboral\n'

2. Configuración de Entorno y Librerías

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro, spearmanr, kruskal, f_oneway, chi2_contingency, pointbiserialr, mannwhitneyu, ttest_ind, fisher_exact
import warnings
import os
from datetime import datetime

# Configuración de estilo y warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuración de rutas
class Config:
    DATA_PATH = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"
    OUTPUT_BASE = r"G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results"
    
    @property
    def OUTPUT_PATH(self):
        return os.path.join(self.OUTPUT_BASE, "Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3")

config = Config()

3. Clases de Utilidades para Análisis Estadístico

In [22]:
class StatisticalAnalyzer:
    """Clase para realizar análisis estadísticos de manera estructurada"""
    
    @staticmethod
    def clasificar_correlacion(corr):
        """Clasificar la fuerza de la correlación según criterios estándar"""
        abs_corr = abs(corr)
        if abs_corr < 0.1:
            return "Muy débil", 1
        elif abs_corr < 0.3:
            return "Débil", 2
        elif abs_corr < 0.5:
            return "Moderada", 3
        elif abs_corr < 0.7:
            return "Fuerte", 4
        else:
            return "Muy fuerte", 5
    
    @staticmethod
    def cohens_d(x, y):
        """Calcular tamaño del efecto Cohen's d"""
        nx, ny = len(x), len(y)
        dof = nx + ny - 2
        pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
        return (np.mean(x) - np.mean(y)) / pooled_std
    
    @staticmethod
    def clasificar_efecto_cohen(d):
        """Clasificar el tamaño del efecto de Cohen"""
        abs_d = abs(d)
        if abs_d < 0.2:
            return "Muy pequeño"
        elif abs_d < 0.5:
            return "Pequeño"
        elif abs_d < 0.8:
            return "Mediano"
        else:
            return "Grande"

class DataValidator:
    """Clase para validaciones de datos"""
    
    @staticmethod
    def verificar_variables(df, variables_requeridas):
        """Verificar que las variables requeridas existen en el DataFrame"""
        faltantes = [var for var in variables_requeridas if var not in df.columns]
        if faltantes:
            raise ValueError(f"Variables faltantes en el dataset: {faltantes}")
        print("✓ Todas las variables requeridas están presentes")
        
    @staticmethod
    def verificar_tamanio_muestral(grupo1, grupo2, min_tamano=5):
        """Verificar tamaño muestral mínimo para análisis"""
        if len(grupo1) < min_tamano or len(grupo2) < min_tamano:
            return False
        return True
    
    @staticmethod
    def verificar_binarias(df, binary_vars):
        """Verificar que las variables binarias contienen solo 0 y 1"""
        print("\n🔍 Validación de variables binarias:")
        for var in binary_vars:
            unique_vals = sorted(df[var].unique())
            if set(unique_vals) <= {0, 1}:
                print(f"  ✓ {var}: binaria correcta [valores: {unique_vals}]")
            else:
                print(f"  ⚠ {var}: NO es estrictamente binaria [valores: {unique_vals}]")
    
    @staticmethod
    def calidad_datos_report(df):
        """Generar reporte completo de calidad de datos"""
        print("\n📊 REPORTE DE CALIDAD DE DATOS:")
        print(f"  • Número de observaciones: {len(df):,}")
        print(f"  • Número de variables: {len(df.columns)}")
        print(f"  • Valores faltantes totales: {df.isnull().sum().sum()}")
        print(f"  • Filas duplicadas: {df.duplicated().sum()}")
        
        # Variables constantes
        constant_vars = [col for col in df.columns if df[col].nunique() == 1]
        if constant_vars:
            print(f"  ⚠ Variables constantes: {constant_vars}")
        else:
            print("  ✓ No hay variables constantes")
        
        # Valores faltantes por variable
        missing_per_var = df.isnull().sum()
        if missing_per_var.any():
            print("\n  🔍 Valores faltantes por variable:")
            for var, count in missing_per_var[missing_per_var > 0].items():
                print(f"    {var}: {count} ({count/len(df)*100:.2f}%)")
        else:
            print("  ✓ No hay valores faltantes")

class BinaryVariablesAnalyzerCorregido:
    """Analizador corregido para variables binarias con enfoques apropiados"""
    
    @staticmethod
    def analizar_binarias_corregido(df, binary_vars, target_cont, target_bin):
        """Análisis corregido para variables binarias usando enfoques apropiados"""
        print("\n🔬 ANÁLISIS CORREGIDO DE VARIABLES BINARIAS:")
        resultados_bin = []
        
        for var in binary_vars:
            if var not in df.columns:
                continue
                
            try:
                # Datos para grupos
                grupo_0 = df[df[var] == 0][target_cont].dropna()
                grupo_1 = df[df[var] == 1][target_cont].dropna()
                
                if len(grupo_0) < 5 or len(grupo_1) < 5:
                    print(f"  ⚠ {var}: Tamaño muestral insuficiente")
                    continue
                
                print(f"  🔍 Analizando {var}:")
                print(f"    - Grupo 0 (n={len(grupo_0)}): media={grupo_0.mean():.2f}")
                print(f"    - Grupo 1 (n={len(grupo_1)}): media={grupo_1.mean():.2f}")
                
                # ENFOQUE 1: Como variable categórica (comparación de medias)
                # ------------------------------------------------------------
                
                # Test de normalidad de los residuos (para decidir test paramétrico vs no paramétrico)
                normalidad_grupo0 = shapiro(grupo_0)[1] > 0.05 if len(grupo_0) >= 3 else False
                normalidad_grupo1 = shapiro(grupo_1)[1] > 0.05 if len(grupo_1) >= 3 else False
                varianzas_iguales = stats.levene(grupo_0, grupo_1)[1] > 0.05
                
                # Decidir test apropiado
                if normalidad_grupo0 and normalidad_grupo1 and varianzas_iguales:
                    # T-test para muestras independientes
                    t_stat, t_p = ttest_ind(grupo_0, grupo_1, equal_var=True)
                    test_utilizado = "T-test"
                    estadistico = t_stat
                    p_valor = t_p
                else:
                    # Mann-Whitney U test (no paramétrico)
                    mw_stat, mw_p = mannwhitneyu(grupo_0, grupo_1, alternative='two-sided')
                    test_utilizado = "Mann-Whitney"
                    estadistico = mw_stat
                    p_valor = mw_p
                
                # Tamaño del efecto (Cohen's d)
                d_effect = StatisticalAnalyzer.cohens_d(grupo_0, grupo_1)
                tamaño_efecto = StatisticalAnalyzer.clasificar_efecto_cohen(d_effect)
                
                # ENFOQUE 2: Como variable numérica (correlación punto-biserial) 
                # -------------------------------------------------------------
                try:
                    # Filtrar datos completos para correlación
                    datos_completos = df[[var, target_cont]].dropna()
                    if len(datos_completos) >= 3:
                        pointbiserial_corr, pointbiserial_p = pointbiserialr(datos_completos[var], datos_completos[target_cont])
                        fuerza_pb, _ = StatisticalAnalyzer.clasificar_correlacion(pointbiserial_corr)
                    else:
                        pointbiserial_corr, pointbiserial_p, fuerza_pb = np.nan, np.nan, "N/A"
                except Exception as e:
                    print(f"    ⚠ Error en correlación punto-biserial para {var}: {e}")
                    pointbiserial_corr, pointbiserial_p, fuerza_pb = np.nan, np.nan, "Error"
                
                # ENFOQUE 3: Asociación con variable binaria objetivo (Odds Ratio) 
                # ---------------------------------------------------------------
                try:
                    tabla_contingencia = pd.crosstab(df[var], df[target_bin])
                    
                    # Asegurar que la tabla sea 2x2 y esté ordenada correctamente
                    if tabla_contingencia.shape == (2, 2):
                        # Reindexar para asegurar orden 0,1
                        tabla_contingencia = tabla_contingencia.reindex(index=[0, 1], columns=[0, 1])
                        
                        # Extraer valores en orden correcto
                        a = tabla_contingencia.iloc[0, 0]  # var=0, target=0
                        b = tabla_contingencia.iloc[0, 1]  # var=0, target=1  
                        c = tabla_contingencia.iloc[1, 0]  # var=1, target=0
                        d = tabla_contingencia.iloc[1, 1]  # var=1, target=1
                        
                        print(f"    - Tabla contingencia corregida: [{a}, {b}; {c}, {d}]")
                        
                        # CÁLCULOS CORRECTOS DE RISK RATIO
                        riesgo_grupo0 = b / (a + b) if (a + b) > 0 else 0
                        riesgo_grupo1 = d / (c + d) if (c + d) > 0 else 0
                        risk_ratio = riesgo_grupo1 / riesgo_grupo0 if riesgo_grupo0 > 0 else np.nan
                        
                        # CÁLCULOS CORRECTOS DE ODDS RATIO
                        odds_grupo0 = b / a if a > 0 else np.nan
                        odds_grupo1 = d / c if c > 0 else np.nan
                        odds_ratio = odds_grupo1 / odds_grupo0 if not np.isnan(odds_grupo1) and not np.isnan(odds_grupo0) else np.nan
                        
                        # Test exacto de Fisher
                        odds_ratio_fisher, fisher_p = fisher_exact(tabla_contingencia)
                        
                        print(f"    - Risk Ratio CORREGIDO: {risk_ratio:.3f}")
                        print(f"    - Odds Ratio CORREGIDO: {odds_ratio:.3f}")
                        
                    else:
                        odds_ratio, odds_ratio_fisher, fisher_p, risk_ratio = np.nan, np.nan, np.nan, np.nan
                        print(f"    ⚠ Tabla de contingencia no es 2x2: {tabla_contingencia.shape}")
                        
                except Exception as e:
                    odds_ratio, odds_ratio_fisher, fisher_p, risk_ratio = np.nan, np.nan, np.nan, np.nan
                    print(f"    ⚠ Error en tabla de contingencia para {var}: {e}")
                
                # Resultados consolidados
                resultados_bin.append({
                    'Variable': var,
                    'Test_Utilizado': test_utilizado,
                    'Estadistico_Test': round(estadistico, 3),
                    'p_valor': round(p_valor, 6),
                    'Significativa': p_valor < 0.05,
                    'Media_Grupo0': round(grupo_0.mean(), 3),
                    'Media_Grupo1': round(grupo_1.mean(), 3),
                    'Diferencia_Medias': round(grupo_1.mean() - grupo_0.mean(), 3),
                    'Cohens_d': round(d_effect, 3),
                    'Tamaño_Efecto': tamaño_efecto,
                    'PointBiserial_Corr': round(pointbiserial_corr, 3) if not np.isnan(pointbiserial_corr) else np.nan,
                    'PointBiserial_p': round(pointbiserial_p, 6) if not np.isnan(pointbiserial_p) else np.nan,
                    'Fuerza_PointBiserial': fuerza_pb,
                    'Odds_Ratio': round(odds_ratio, 3) if not np.isnan(odds_ratio) else np.nan,
                    'Fisher_p': round(fisher_p, 6) if not np.isnan(fisher_p) else np.nan,
                    'Risk_Ratio': round(risk_ratio, 3) if not np.isnan(risk_ratio) else np.nan,
                    'Interpretacion_Direccion': "AUMENTA" if (grupo_1.mean() > grupo_0.mean()) else "DISMINUYE"
                })
                
                print(f"    ✓ Análisis completado: p={p_valor:.6f}, Cohen's d={d_effect:.3f}")
                
            except Exception as e:
                print(f"  ✗ Error analizando {var}: {e}")
        
        return pd.DataFrame(resultados_bin)
    
    @staticmethod
    def interpretar_resultados_binarios(df_results_bin):
        """Interpretación mejorada de resultados binarios"""
        if df_results_bin.empty:
            print("  No hay resultados para interpretar")
            return
            
        print("\n📋 INTERPRETACIÓN CORREGIDA - VARIABLES BINARIAS:")
        print("="*60)
        
        for _, row in df_results_bin.iterrows():
            if row['Significativa']:
                # Interpretación basada en diferencia de medias
                efecto = row['Interpretacion_Direccion']
                print(f"\n  🎯 {row['Variable']}:")
                print(f"     • Efecto: {efecto} el ausentismo ({row['Test_Utilizado']}, p={row['p_valor']:.4f})")
                print(f"     • Diferencia: {row['Media_Grupo1']:.2f} vs {row['Media_Grupo0']:.2f} horas")
                print(f"     • Tamaño efecto: Cohen's d = {row['Cohens_d']:.2f} ({row['Tamaño_Efecto']})")
                
                # Interpretación punto-biserial si está disponible
                if not np.isnan(row['PointBiserial_Corr']):
                    print(f"     • Correlación punto-biserial: {row['PointBiserial_Corr']:.3f} ({row['Fuerza_PointBiserial']})")
                
                # Interpretación odds ratio si está disponible
                if not np.isnan(row['Odds_Ratio']):
                    interpretacion_or = "aumenta riesgo" if row['Odds_Ratio'] > 1 else "disminuye riesgo"
                    print(f"     • Odds Ratio: {row['Odds_Ratio']:.2f} ({interpretacion_or})")
                
                # Interpretación risk ratio si está disponible
                if not np.isnan(row['Risk_Ratio']):
                    interpretacion_rr = "aumenta riesgo" if row['Risk_Ratio'] > 1 else "disminuye riesgo"
                    print(f"     • Risk Ratio: {row['Risk_Ratio']:.2f} ({interpretacion_rr})")
            else:
                print(f"\n  📊 {row['Variable']}: No significativa (p={row['p_valor']:.4f})")

4. Carga y Validación de Datos

In [23]:
print("="*80)
print("FASE 1: CARGA Y PREPARACIÓN DE DATOS")
print("="*80)

try:
    # Cargar datos
    df = pd.read_parquet(config.DATA_PATH)
    print(f"✓ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    # Definir variables con clasificación CORREGIDA
    VARIABLES_BINARIAS = ['Disciplinary_failure', 'Social_drinker', 'Social_smoker']

    # Variables personales (EXCLUYENDO Reason_absence_numeric porque es categórica)
    VARIABLES_PERSONALES = [
        'Age', 'Son', 'Pet', 'Weight', 'Height', 
        'Body_mass_index', 'Education_numeric', 'Social_drinker', 'Social_smoker'
        # Reason_absence_numeric se quita porque es categórica
    ]
    
    # Variables laborales
    VARIABLES_LABORALES = [
        'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
        'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure'
    ]
    
    # Variables temporales (EXCLUIDAS del análisis principal)
    VARIABLES_TEMPORALES = [
        'Month_absence', 'Day_week', 'Seasons', 'Reason_absence'  # Versiones categóricas con strings
    ]
    
    # Variables temporales numéricas (para análisis específico si se necesita)
    VARIABLES_TEMPORALES_NUMERICAS = [
        'Month_absence_order', 'Day_week_order', 'Seasons_order'
    ]
    
    # CORRECCIÓN: VARIABLES_NUMERICAS solo debe incluir variables verdaderamente numéricas
    VARIABLES_NUMERICAS = VARIABLES_PERSONALES + VARIABLES_LABORALES + VARIABLES_TEMPORALES_NUMERICAS

    # Variables categóricas: incluir Reason_absence_numeric como categórica y también la versión agrupada
    VARIABLES_CATEGORICAS = ['Reason_absence', 'Month_absence', 'Day_week', 'Seasons', 'Education'] + VARIABLES_BINARIAS
    TARGET_CONTINUO = 'Absenteeism_hours'
    
    # Validar variables
    todas_variables = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS + [TARGET_CONTINUO]
    DataValidator.verificar_variables(df, todas_variables)
    
    # Validaciones específicas
    DataValidator.verificar_binarias(df, VARIABLES_BINARIAS)
    DataValidator.calidad_datos_report(df)
    
    # Preparar datos - CREAR 3 CATEGORÍAS DE AUSENTISMO
    df_clean = df.drop(columns=['ID'], errors='ignore')
    
    # Crear 3 categorías de ausentismo (33%, 33-66%, >66%)
    percentil_33 = df_clean[TARGET_CONTINUO].quantile(0.33)
    percentil_66 = df_clean[TARGET_CONTINUO].quantile(0.66)
    
    def categorizar_ausentismo(horas):
        if horas <= percentil_33:
            return 'Bajo'
        elif horas <= percentil_66:
            return 'Medio'
        else:
            return 'Alto'
    
    df_clean['Ausentismo_Categoria'] = df_clean[TARGET_CONTINUO].apply(categorizar_ausentismo)
    df_clean['Ausentismo_Alto'] = (df_clean[TARGET_CONTINUO] > percentil_66).astype(int)
    TARGET_BINARIO = 'Ausentismo_Alto'
    TARGET_CATEGORICO = 'Ausentismo_Categoria'
    
    # CREAR VARIABLE AGRUPADA PARA RAZONES DE AUSENTISMO
    reason_group_dict = {
        "Enfermedades físicas": [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14],
        "Salud mental": [5],
        "Salud reproductiva": [15, 16, 17],
        "Condiciones diversas": [18, 19, 20, 21],
        "Procedimientos médicos": [22, 24, 25, 27, 28],
        "Consulta médica": [23],
        "Ausencias no justificadas": [26],
        "Otros": [0]
    }
    
    # Mapear Reason_absence_numeric a grupos
    def map_reason_group(codigo):
        for grupo, codigos in reason_group_dict.items():
            if codigo in codigos:
                return grupo
        return 'Otros'
    
    df_clean['Razon_agrupada'] = df_clean['Reason_absence_numeric'].apply(map_reason_group)
    
    # Añadir Razon_agrupada a las variables categóricas
    VARIABLES_CATEGORICAS.append('Razon_agrupada')
    
    print(f"✓ Datos preparados. Categorías de ausentismo:")
    print(f"  - Bajo: ≤ {percentil_33:.2f} horas (33% inferior)")
    print(f"  - Medio: {percentil_33:.2f} - {percentil_66:.2f} horas")
    print(f"  - Alto: > {percentil_66:.2f} horas (33% superior)")
    print(f"✓ Variables creadas: {TARGET_BINARIO} (binario), {TARGET_CATEGORICO} (categórico)")
    print(f"✓ Variables personales: {len(VARIABLES_PERSONALES)}")
    print(f"✓ Variables laborales: {len(VARIABLES_LABORALES)}")
    print(f"✓ Variables temporales (excluidas): {len(VARIABLES_TEMPORALES)}")
    print(f"✓ Variables numéricas totales: {len(VARIABLES_NUMERICAS)}")
    print(f"✓ Variable Razon_agrupada creada para análisis categórico")
    
    # Verificación de tipos de datos
    print(f"\n🔍 VERIFICACIÓN DE TIPOS DE DATOS EN VARIABLES_NUMERICAS:")
    for var in VARIABLES_NUMERICAS:
        if var in df_clean.columns:
            dtype = df_clean[var].dtype
            print(f"  ✓ {var}: {dtype}")
    
except Exception as e:
    print(f"✗ Error en carga de datos: {e}")
    raise

FASE 1: CARGA Y PREPARACIÓN DE DATOS
✓ Dataset cargado: 806 filas, 26 columnas
✓ Todas las variables requeridas están presentes

🔍 Validación de variables binarias:
  ✓ Disciplinary_failure: binaria correcta [valores: [np.int64(0), np.int64(1)]]
  ✓ Social_drinker: binaria correcta [valores: [np.int64(0), np.int64(1)]]
  ✓ Social_smoker: binaria correcta [valores: [np.int64(0), np.int64(1)]]

📊 REPORTE DE CALIDAD DE DATOS:
  • Número de observaciones: 806
  • Número de variables: 26
  • Valores faltantes totales: 10
  • Filas duplicadas: 0
  ✓ No hay variables constantes

  🔍 Valores faltantes por variable:
    Month_absence: 5 (0.62%)
    Month_absence_order: 5 (0.62%)
✓ Datos preparados. Categorías de ausentismo:
  - Bajo: ≤ 2.00 horas (33% inferior)
  - Medio: 2.00 - 8.00 horas
  - Alto: > 8.00 horas (33% superior)
✓ Variables creadas: Ausentismo_Alto (binario), Ausentismo_Categoria (categórico)
✓ Variables personales: 9
✓ Variables laborales: 6
✓ Variables temporales (excluidas): 4

4. 5. ANÁLISIS DE CALIDAD DE DATOS

5. Análisis de Normalidad

In [24]:
print("\n" + "="*80)
print("FASE 5: ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)")
print("="*80)

def analizar_normalidad(df, variables):
    """Realizar test de normalidad Shapiro-Wilk para múltiples variables"""
    resultados = []
    
    # Filtrar solo variables numéricas
    variables_numericas_reales = [var for var in variables if pd.api.types.is_numeric_dtype(df[var])]
    variables_no_numericas = [var for var in variables if not pd.api.types.is_numeric_dtype(df[var])]
    
    if variables_no_numericas:
        print(f"⚠ Variables no numéricas excluidas del análisis de normalidad: {variables_no_numericas}")
    
    for var in variables_numericas_reales:
        data = df[var].dropna()
        if len(data) > 3:  # Mínimo requerido para Shapiro-Wilk
            try:
                stat, p_value = shapiro(data)
                resultados.append({
                    'Variable': var,
                    'Estadistico_Shapiro': round(stat, 6),
                    'p_value_Shapiro': round(p_value, 6),
                    'Es_Normal': p_value > 0.05,
                    'n': len(data)
                })
            except Exception as e:
                print(f"⚠ Error en test de normalidad para {var}: {e}")
        else:
            print(f"⚠ Variable {var} no tiene suficientes datos para análisis de normalidad (n={len(data)})")
    
    return pd.DataFrame(resultados)

# Crear lista CORREGIDA de variables para análisis de normalidad
# Solo incluir variables personales, laborales y el target (excluyendo variables temporales categóricas y Reason_absence_numeric)
variables_para_normalidad = VARIABLES_PERSONALES + VARIABLES_LABORALES + [TARGET_CONTINUO]

# Asegurarnos de que Month_absence_order esté incluida si existe
if 'Month_absence_order' in df_clean.columns and 'Month_absence_order' not in variables_para_normalidad:
    variables_para_normalidad.append('Month_absence_order')

print("🔍 Variables incluidas en análisis de normalidad:")
print(f"  - Personales: {len(VARIABLES_PERSONALES)} variables")
print(f"  - Laborales: {len(VARIABLES_LABORALES)} variables")
print(f"  - Target: 1 variable")
print(f"  - Total: {len(variables_para_normalidad)} variables")
print(f"  - EXCLUIDA: Reason_absence_numeric (variable categórica ordinal)")

# Verificación adicional de tipos de datos
print(f"\n🔍 VERIFICACIÓN DE TIPOS DE DATOS PARA ANÁLISIS DE NORMALIDAD:")
variables_numericas_verificadas = []
variables_excluidas = []

for var in variables_para_normalidad:
    if var in df_clean.columns:
        if pd.api.types.is_numeric_dtype(df_clean[var]):
            n_missing = df_clean[var].isnull().sum()
            unique_vals = df_clean[var].dropna().unique()
            print(f"  ✓ {var}: numérica (tipo: {df_clean[var].dtype}, faltantes: {n_missing}, valores únicos: {len(unique_vals)})")
            variables_numericas_verificadas.append(var)
        else:
            print(f"  ✗ {var}: NO numérica (tipo: {df_clean[var].dtype}) - EXCLUIDA")
            variables_excluidas.append(var)
    else:
        print(f"  ⚠ {var}: no encontrada en el dataset")
        variables_excluidas.append(var)

print(f"\n📋 RESUMEN:")
print(f"  - Variables verificadas: {len(variables_numericas_verificadas)}")
print(f"  - Variables excluidas: {len(variables_excluidas)}")

# Ejecutar análisis solo con variables numéricas verificadas
df_normality = analizar_normalidad(df_clean, variables_numericas_verificadas)

if not df_normality.empty:
    print("\n📊 RESULTADOS DE NORMALIDAD (Shapiro-Wilk):")
    print(df_normality.to_string(index=False))
    
    # Resumen estadístico
    normales = df_normality['Es_Normal'].sum()
    total_vars = len(df_normality)
    print(f"\n📊 RESUMEN ESTADÍSTICO:")
    print(f"  - Variables analizadas: {total_vars}")
    print(f"  - Variables normales: {normales} ({normales/total_vars*100:.1f}%)")
    print(f"  - Variables no normales: {total_vars - normales} ({(total_vars - normales)/total_vars*100:.1f}%)")
    
    # Variables no normales (para referencia)
    variables_no_normales = df_normality[~df_normality['Es_Normal']]['Variable'].tolist()
    if variables_no_normales:
        print(f"  📋 Variables NO normales: {', '.join(variables_no_normales)}")
    
    # Implicaciones para el análisis
    print(f"\n💡 IMPLICACIONES ESTADÍSTICAS:")
    if normales == total_vars:
        print("  - TODAS las variables son normales → Se pueden usar tests PARAMÉTRICOS")
    elif normales == 0:
        print("  - NINGUNA variable es normal → Se usarán tests NO PARAMÉTRICOS")
        print("  - Tests a usar: Spearman, Mann-Whitney, Kruskal-Wallis")
    else:
        print("  - ALGUNAS variables no son normales → Se recomiendan tests NO PARAMÉTRICOS")
        print("  - Tests a usar: Spearman, Mann-Whitney, Kruskal-Wallis")
        
    print(f"\n🎯 DECISIÓN FINAL: Se usarán tests NO PARAMÉTRICOS para mayor robustez")
    
else:
    print("❌ No se pudo realizar el análisis de normalidad - no hay variables numéricas válidas")
    # Crear un DataFrame vacío con la estructura esperada para evitar errores posteriores
    df_normality = pd.DataFrame(columns=['Variable', 'Estadistico_Shapiro', 'p_value_Shapiro', 'Es_Normal', 'n'])

# Información adicional sobre variables temporales (para referencia)
print(f"\n📌 NOTA SOBRE VARIABLES TEMPORALES:")
variables_temporales_numericas = [var for var in VARIABLES_TEMPORALES if var in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean[var])]
variables_temporales_categoricas = [var for var in VARIABLES_TEMPORALES if var in df_clean.columns and not pd.api.types.is_numeric_dtype(df_clean[var])]

if variables_temporales_numericas:
    print(f"  - Variables temporales numéricas (no incluidas en análisis principal): {variables_temporales_numericas}")
if variables_temporales_categoricas:
    print(f"  - Variables temporales categóricas: {variables_temporales_categoricas}")


FASE 5: ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)
🔍 Variables incluidas en análisis de normalidad:
  - Personales: 9 variables
  - Laborales: 6 variables
  - Target: 1 variable
  - Total: 17 variables
  - EXCLUIDA: Reason_absence_numeric (variable categórica ordinal)

🔍 VERIFICACIÓN DE TIPOS DE DATOS PARA ANÁLISIS DE NORMALIDAD:
  ✓ Age: numérica (tipo: float64, faltantes: 0, valores únicos: 22)
  ✓ Son: numérica (tipo: int64, faltantes: 0, valores únicos: 5)
  ✓ Pet: numérica (tipo: int64, faltantes: 0, valores únicos: 6)
  ✓ Weight: numérica (tipo: float64, faltantes: 0, valores únicos: 26)
  ✓ Height: numérica (tipo: float64, faltantes: 0, valores únicos: 14)
  ✓ Body_mass_index: numérica (tipo: float64, faltantes: 0, valores únicos: 17)
  ✓ Education_numeric: numérica (tipo: int64, faltantes: 0, valores únicos: 4)
  ✓ Social_drinker: numérica (tipo: int64, faltantes: 0, valores únicos: 2)
  ✓ Social_smoker: numérica (tipo: int64, faltantes: 0, valores únicos: 2)
  ✓ Transportation_exp

6. Análisis de Variables Numéricas

In [25]:
print("\n" + "="*80)
print("FASE 6: ANÁLISIS DE VARIABLES NUMÉRICAS")
print("="*80)

def analizar_variables_numericas(df, variables, target_cont, target_bin, df_normality):
    """Análisis completo para variables numéricas"""
    resultados = []
    analyzer = StatisticalAnalyzer()
    
    for i, var in enumerate(variables, 1):
        print(f"🔍 Analizando {i}/{len(variables)}: {var}")
        
        if var not in df.columns:
            print(f"  ⚠ Variable {var} no encontrada, saltando...")
            continue
            
        # EXCLUIR Reason_absence_numeric del análisis numérico
        if var == 'Reason_absence_numeric':
            print(f"  ⚠ {var} EXCLUIDA - es variable categórica ordinal")
            continue
            
        # Preparar datos para grupos
        data_alto = df[df[target_bin] == 1][var].dropna()
        data_bajo = df[df[target_bin] == 0][var].dropna()
        
        # Verificar tamaño muestral
        if not DataValidator.verificar_tamanio_muestral(data_alto, data_bajo):
            print(f"  ⚠ Tamaño muestral insuficiente para {var}, saltando...")
            continue
        
        try:
            # Correlación Spearman con manejo robusto de datos
            data_var = df[var].dropna()
            data_target = df[target_cont].dropna()
            
            # Alinear índices para evitar error de dimensiones
            common_idx = data_var.index.intersection(data_target.index)
            if len(common_idx) < 3:
                print(f"  ⚠ Datos insuficientes para correlación en {var}")
                continue
                
            spearman_corr, spearman_p = spearmanr(data_var.loc[common_idx], data_target.loc[common_idx])
            fuerza_spearman, fuerza_num = analyzer.clasificar_correlacion(spearman_corr)
            
            # Point-biserial
            pointbiserial_corr, pointbiserial_p = pointbiserialr(df[var].dropna(), df[target_bin].dropna())
            
            # Tests de diferencias
            mw_stat, mw_p = mannwhitneyu(data_alto, data_bajo, alternative='two-sided')
            t_stat, t_p = ttest_ind(data_alto, data_bajo, equal_var=False)
            
            # Tamaño del efecto
            d_effect = analyzer.cohens_d(data_alto, data_bajo)
            tamaño_efecto = analyzer.clasificar_efecto_cohen(d_effect)
            
            # Determinar test apropiado
            es_normal_var = df_normality[df_normality['Variable'] == var]['Es_Normal'].values[0]
            test_medias = "T-test" if es_normal_var else "Mann-Whitney"
            p_value_medias = t_p if es_normal_var else mw_p
            
            resultados.append({
                'Variable': var,
                'Correlacion_Spearman': round(spearman_corr, 6),
                'p_value_Spearman': round(spearman_p, 6),
                'Fuerza_Spearman': fuerza_spearman,
                'Significativa_Spearman': spearman_p < 0.05,
                'Cohens_d': round(d_effect, 6),
                'Tamaño_Efecto': tamaño_efecto,
                'Test_Medias': test_medias,
                'p_value_Medias': round(p_value_medias, 6),
                'Significativa_Medias': p_value_medias < 0.05,
                'Media_Alto': round(data_alto.mean(), 3),
                'Media_Bajo': round(data_bajo.mean(), 3),
                'Diferencia_Medias': round(data_alto.mean() - data_bajo.mean(), 3),
                'abs_corr': abs(spearman_corr),
                'abs_effect': abs(d_effect)
            })
            
        except Exception as e:
            print(f"  ⚠ Error en {var}: {e}")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_results_numeric = analizar_variables_numericas(
    df_clean, VARIABLES_NUMERICAS, TARGET_CONTINUO, TARGET_BINARIO, df_normality
)

print("\n📋 Resultados del análisis numérico:")
print(df_results_numeric[['Variable', 'Correlacion_Spearman', 'Fuerza_Spearman', 
                         'Significativa_Spearman', 'Cohens_d', 'Tamaño_Efecto']].to_string(index=False))


FASE 6: ANÁLISIS DE VARIABLES NUMÉRICAS
🔍 Analizando 1/18: Age
🔍 Analizando 2/18: Son
🔍 Analizando 3/18: Pet
🔍 Analizando 4/18: Weight
🔍 Analizando 5/18: Height
🔍 Analizando 6/18: Body_mass_index
🔍 Analizando 7/18: Education_numeric
🔍 Analizando 8/18: Social_drinker
🔍 Analizando 9/18: Social_smoker
🔍 Analizando 10/18: Transportation_expense
🔍 Analizando 11/18: Distance_Residence_Work
🔍 Analizando 12/18: Service_time
🔍 Analizando 13/18: Work_load_Average_day
🔍 Analizando 14/18: Hit_target
🔍 Analizando 15/18: Disciplinary_failure
🔍 Analizando 16/18: Month_absence_order
  ⚠ Error en Month_absence_order: `x` and `y` must have the same length along `axis`.
🔍 Analizando 17/18: Day_week_order
  ⚠ Error en Day_week_order: index 0 is out of bounds for axis 0 with size 0
🔍 Analizando 18/18: Seasons_order
  ⚠ Error en Seasons_order: index 0 is out of bounds for axis 0 with size 0

📋 Resultados del análisis numérico:
               Variable  Correlacion_Spearman Fuerza_Spearman  Significativa_Spe

7. Análisis de Variables Categóricas

In [26]:
print("\n" + "="*80)
print("FASE 7: ANÁLISIS DE VARIABLES CATEGÓRICAS")
print("="*80)

def analizar_variables_categoricas(df, variables, target_cont, target_bin, df_normality):
    """Análisis completo para variables categóricas"""
    resultados = []
    
    for i, var in enumerate(variables, 1):
        print(f"🔍 Analizando {i}/{len(variables)}: {var}")
        
        if var not in df.columns:
            print(f"  ⚠ Variable {var} no encontrada, saltando...")
            continue
            
        try:
            # Kruskal-Wallis
            grupos_kw = [grupo[target_cont].values for nombre, grupo in df.groupby(var) if len(grupo) > 5]
            
            if len(grupos_kw) > 1:
                kw_stat, kw_p = kruskal(*grupos_kw)
            else:
                kw_stat, kw_p = np.nan, np.nan
                
            # Chi-cuadrado
            tabla_contingencia = pd.crosstab(df[var], df[target_bin])
            if tabla_contingencia.shape[0] > 1 and tabla_contingencia.shape[1] > 1:
                chi2_stat, chi2_p, dof, esperado = chi2_contingency(tabla_contingencia)
            else:
                chi2_stat, chi2_p = np.nan, np.nan
            
            # Determinar test principal
            es_normal_target = df_normality[df_normality['Variable'] == target_cont]['Es_Normal'].values[0]
            test_principal = 'ANOVA' if es_normal_target else 'Kruskal-Wallis'
            p_value_principal = kw_p
            
            resultados.append({
                'Variable': var,
                'Test_Principal': test_principal,
                'p_value_Principal': round(p_value_principal, 6) if not np.isnan(p_value_principal) else np.nan,
                'Significativa_Principal': p_value_principal < 0.05 if not np.isnan(p_value_principal) else False,
                'p_value_Chi2': round(chi2_p, 6) if not np.isnan(chi2_p) else np.nan,
                'Significativa_Chi2': chi2_p < 0.05 if not np.isnan(chi2_p) else False,
                'n_Categorias': len(df[var].unique())
            })
            
        except Exception as e:
            print(f"  ✗ Error analizando {var}: {e}")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis categórico
df_results_categorical = analizar_variables_categoricas(
    df_clean, VARIABLES_CATEGORICAS, TARGET_CONTINUO, TARGET_BINARIO, df_normality
)

print("\n📋 Resultados del análisis categórico:")
print(df_results_categorical[['Variable', 'Test_Principal', 'p_value_Principal', 
                            'Significativa_Principal', 'Significativa_Chi2']].to_string(index=False))

# ANÁLISIS CORREGIDO DE VARIABLES BINARIAS
print("\n" + "="*80)
print("ANÁLISIS CORREGIDO DE VARIABLES BINARIAS")
print("="*80)

# Ejecutar análisis corregido
print("🔍 Ejecutando análisis corregido de variables binarias...")
df_results_binarias_corregido = BinaryVariablesAnalyzerCorregido.analizar_binarias_corregido(
    df_clean, VARIABLES_BINARIAS, TARGET_CONTINUO, TARGET_BINARIO
)

# Mostrar resultados
if not df_results_binarias_corregido.empty:
    print("\n📊 RESULTADOS CORREGIDOS - VARIABLES BINARIAS:")
    print("="*60)
    columnas_mostrar = ['Variable', 'Test_Utilizado', 'p_valor', 'Significativa', 
                       'Diferencia_Medias', 'Cohens_d', 'PointBiserial_Corr']
    print(df_results_binarias_corregido[columnas_mostrar].to_string(index=False))
    
    # Interpretación
    BinaryVariablesAnalyzerCorregido.interpretar_resultados_binarios(df_results_binarias_corregido)
else:
    print("❌ No se pudieron generar resultados para variables binarias")

# COMPARACIÓN CON EL ANÁLISIS ANTERIOR (si existe)
if 'df_results_binarias' in locals() and not df_results_binarias.empty:
    print("\n" + "="*80)
    print("COMPARACIÓN: ANÁLISIS ANTERIOR vs CORREGIDO")
    print("="*80)
    
    print("ANÁLISIS ANTERIOR:")
    print(df_results_binarias[['Variable', 'Mann_Whitney_p', 'Significativa_MW', 'Cohens_d']].to_string(index=False))
    
    print("\nANÁLISIS CORREGIDO:")
    print(df_results_binarias_corregido[['Variable', 'Test_Utilizado', 'p_valor', 'Significativa', 'Cohens_d']].to_string(index=False))

# ANÁLISIS ESPECÍFICO DE RAZONES AGRUPADAS
print("\n" + "="*80)
print("ANÁLISIS CORREGIDO DE RAZONES DE AUSENTISMO")
print("="*80)

def analizar_razones_agrupadas(df, target_cont, target_bin, output_path):
    """Análisis específico para la variable Razon_agrupada"""
    print("🔍 ANALIZANDO RAZONES AGRUPADAS:")
    
    # Asegurar que el directorio de salida existe
    os.makedirs(output_path, exist_ok=True)
    print(f"  📁 Directorio verificado: {output_path}")
    
    # Estadísticas descriptivas por grupo de razones
    print("\n📊 DISTRIBUCIÓN DE HORAS POR TIPO DE RAZÓN:")
    stats_razones = df.groupby('Razon_agrupada')[target_cont].agg([
        'count', 'mean', 'median', 'std', 'min', 'max'
    ]).round(2)
    print(stats_razones)
    
    # Kruskal-Wallis test entre grupos
    grupos = [grupo[target_cont].values for nombre, grupo in df.groupby('Razon_agrupada')]
    kw_stat, kw_p = kruskal(*grupos)
    
    print(f"\n🔬 TEST KRUSKAL-WALLIS ENTRE GRUPOS DE RAZONES:")
    print(f"  - Estadístico: {kw_stat:.4f}")
    print(f"  - p-valor: {kw_p:.6f}")
    print(f"  - Significativo: {'SÍ' if kw_p < 0.05 else 'NO'}")
    
    # Chi-cuadrado para asociación con ausentismo alto
    tabla_contingencia = pd.crosstab(df['Razon_agrupada'], df[target_bin])
    chi2_stat, chi2_p, dof, esperado = chi2_contingency(tabla_contingencia)
    
    print(f"\n🔬 TEST CHI-CUADRADO CON AUSENTISMO ALTO:")
    print(f"  - Estadístico: {chi2_stat:.4f}")
    print(f"  - p-valor: {chi2_p:.6f}")
    print(f"  - Significativo: {'SÍ' if chi2_p < 0.05 else 'NO'}")
    
    # Visualización
    plt.figure(figsize=(12, 8))
    
    # Boxplot
    plt.subplot(1, 2, 1)
    orden_razones = df.groupby('Razon_agrupada')[target_cont].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x='Razon_agrupada', y=target_cont, order=orden_razones)
    plt.title('Distribución de Horas por Tipo de Razón')
    plt.xticks(rotation=45, ha='right')
    
    # Porcentaje de ausentismo alto por razón
    plt.subplot(1, 2, 2)
    porcentaje_alto = df.groupby('Razon_agrupada')[target_bin].mean().sort_values(ascending=False)
    sns.barplot(x=porcentaje_alto.values, y=porcentaje_alto.index)
    plt.title('Porcentaje de Ausentismo Alto por Tipo de Razón')
    plt.xlabel('Porcentaje de Ausentismo Alto')
    
    plt.tight_layout()
    
    # Guardar gráfico - asegurando que el directorio existe
    try:
        plt.savefig(f'{output_path}/analisis_razones_agrupadas.png', dpi=300, bbox_inches='tight')
        print(f"✓ Gráfico de análisis de razones guardado: {output_path}/analisis_razones_agrupadas.png")
    except Exception as e:
        print(f"⚠ Error guardando gráfico: {e}")
        # Intentar guardar en ubicación alternativa
        try:
            plt.savefig('analisis_razones_agrupadas_backup.png', dpi=300, bbox_inches='tight')
            print("✓ Gráfico guardado en ubicación alternativa: analisis_razones_agrupadas_backup.png")
        except Exception as e2:
            print(f"✗ Error grave guardando gráfico: {e2}")
    
    plt.close()
    
    return {
        'kw_statistic': kw_stat,
        'kw_pvalue': kw_p,
        'chi2_statistic': chi2_stat,
        'chi2_pvalue': chi2_p,
        'stats_por_razon': stats_razones
    }

# Asegurar que el directorio de salida existe antes de ejecutar el análisis
print(f"🔧 Verificando directorio de salida: {config.OUTPUT_PATH}")
os.makedirs(config.OUTPUT_PATH, exist_ok=True)
print(f"✓ Directorio listo: {config.OUTPUT_PATH}")

# Ejecutar análisis de razones agrupadas
resultados_razones = analizar_razones_agrupadas(df_clean, TARGET_CONTINUO, TARGET_BINARIO, config.OUTPUT_PATH)


FASE 7: ANÁLISIS DE VARIABLES CATEGÓRICAS
🔍 Analizando 1/9: Reason_absence
🔍 Analizando 2/9: Month_absence
🔍 Analizando 3/9: Day_week
🔍 Analizando 4/9: Seasons
🔍 Analizando 5/9: Education
🔍 Analizando 6/9: Disciplinary_failure
🔍 Analizando 7/9: Social_drinker
🔍 Analizando 8/9: Social_smoker
🔍 Analizando 9/9: Razon_agrupada

📋 Resultados del análisis categórico:
            Variable Test_Principal  p_value_Principal  Significativa_Principal  Significativa_Chi2
      Reason_absence Kruskal-Wallis           0.000000                     True                True
       Month_absence Kruskal-Wallis           0.001513                     True               False
            Day_week Kruskal-Wallis           0.020356                     True                True
             Seasons Kruskal-Wallis           0.021905                     True               False
           Education Kruskal-Wallis           0.423551                    False               False
Disciplinary_failure Kruskal-Wallis

In [27]:
# Diccionari de motius
reason_absence_dict = {
    0: "Sin especificar / Otros",
    1: "Enfermedades infecciosas y parasitarias",
    2: "Neoplasias (tumores)",
    3: "Enfermedades de la sangre y órganos hematopoyéticos",
    4: "Enfermedades endocrinas, nutricionales y metabólicas",
    5: "Trastornos mentales y del comportamiento",
    6: "Enfermedades del sistema nervioso",
    7: "Enfermedades del ojo y sus anexos",
    8: "Enfermedades del oído y la apófisis mastoides",
    9: "Enfermedades del sistema circulatorio",
    10: "Enfermedades del sistema respiratorio",
    11: "Enfermedades del sistema digestivo",
    12: "Enfermedades de la piel y tejido subcutáneo",
    13: "Enfermedades del sistema musculoesquelético y tejido conectivo",
    14: "Enfermedades del sistema genitourinario",
    15: "Embarazo, parto y puerperio",
    16: "Condiciones originadas en el período perinatal",
    17: "Malformaciones congénitas y anomalías cromosómicas",
    18: "Síntomas y signos no clasificados",
    19: "Lesiones, envenenamientos y consecuencias de otras causas externas",
    20: "Causas externas de morbilidad y mortalidad",
    21: "Factores que influyen en el estado de salud",
    22: "Paciente en seguimiento",
    23: "Consulta médica",
    24: "Donación de sangre",
    25: "Examen de laboratorio",
    26: "Ausencia injustificada",
    27: "Fisioterapia",
    28: "Consulta dental"
}
# Diccionari simplificat per a agrupacions més generals
reason_group_dict  = {
        # GRUPOS ESPECÍFICOS (alta importancia)
        "Salud mental": [5],
        "Problemas musculoesqueléticos": [13],
        "Problemas digestivos": [11],
        "Salud reproductiva": [15, 16, 17],
        "Ausencias no justificadas": [26],
        
        # GRUPOS MODERADOS (balance ideal)
        "Enfermedades respiratorias": [8, 1, 2],
        "Problemas cardiovasculares": [3, 4],
        "Condiciones neurológicas": [6, 9],
        
        # GRUPOS AMPLIOS (poder estadístico)
        "Otras enfermedades físicas": [7, 10, 12, 14],
        "Consultas médicas": [23, 22, 24],
        "Procedimientos médicos": [25, 27, 28],
        "Condiciones diversas": [18, 19, 20, 21],
        "Otros/Sin especificar": [0]
    }

7. 5: VERIFICACIÓN Y CORRECCIÓN DE MÉTRICAS BINARIAS

In [28]:
print("\n" + "="*80)
print("FASE 7.5: VERIFICACIÓN Y CORRECCIÓN DE MÉTRICAS BINARIAS")
print("="*80)

# Asegurar que el directorio de salida existe ANTES de cualquier operación de guardado
import os
os.makedirs(config.OUTPUT_PATH, exist_ok=True)
print(f"📁 Directorio de salida verificado: {config.OUTPUT_PATH}")

class MetricasCorrector:
    """Clase para verificar y corregir cálculos de OR, RR y validar variables"""
    
    def __init__(self, output_path):
        self.output_path = output_path
        # Asegurar que el directorio existe
        os.makedirs(output_path, exist_ok=True)
    
    @staticmethod
    def verificar_codificacion_disciplinary_failure(df):
        """Verificación detallada de Disciplinary_failure"""
        print("\n🔍 VALIDACIÓN MANUAL DE DISCIPLINARY_FAILURE:")
        print("-" * 50)
        
        if 'Disciplinary_failure' not in df.columns:
            print("❌ Disciplinary_failure no encontrada en el dataset")
            return
        
        # Análisis descriptivo completo
        print("📊 DISTRIBUCIÓN DE DISCIPLINARY_FAILURE:")
        print(df['Disciplinary_failure'].value_counts())
        print(f"Proporciones: {df['Disciplinary_failure'].value_counts(normalize=True)}")
        
        print("\n📈 AUSENTISMO POR DISCIPLINARY_FAILURE:")
        ausentismo_por_grupo = df.groupby('Disciplinary_failure')['Absenteeism_hours'].agg([
            'count', 'mean', 'median', 'std', 'min', 'max'
        ]).round(2)
        print(ausentismo_por_grupo)
        
        # Tabla de contingencia detallada
        print("\n🎯 TABLA DE CONTINGENCIA COMPLETA:")
        tabla = pd.crosstab(df['Disciplinary_failure'], df['Ausentismo_Alto'], 
                           margins=True, margins_name="Total")
        print(tabla)
        
        # Cálculo manual de OR y RR
        print("\n🧮 CÁLCULO MANUAL DE OR Y RR:")
        if tabla.shape == (3, 3):  # Incluye totales
            a = tabla.iloc[0, 0]  # Grupo 0, Ausentismo Bajo
            b = tabla.iloc[0, 1]  # Grupo 0, Ausentismo Alto
            c = tabla.iloc[1, 0]  # Grupo 1, Ausentismo Bajo
            d = tabla.iloc[1, 1]  # Grupo 1, Ausentismo Alto
            
            # Odds Ratio manual
            odds_ratio_manual = (d / c) / (b / a) if (a > 0 and c > 0) else np.nan
            
            # Risk Ratio manual
            riesgo_grupo1 = d / (c + d) if (c + d) > 0 else 0
            riesgo_grupo0 = b / (a + b) if (a + b) > 0 else 0
            risk_ratio_manual = riesgo_grupo1 / riesgo_grupo0 if riesgo_grupo0 > 0 else np.nan
            
            print(f"  - Odds Ratio manual: {odds_ratio_manual:.4f}")
            print(f"  - Risk Ratio manual: {risk_ratio_manual:.4f}")
            print(f"  - Riesgo Grupo 0: {riesgo_grupo0:.4f}")
            print(f"  - Riesgo Grupo 1: {riesgo_grupo1:.4f}")
            
            # Interpretación
            if not np.isnan(odds_ratio_manual):
                if odds_ratio_manual > 1:
                    print("  📈 Interpretación OR: AUMENTA el riesgo de ausentismo alto")
                else:
                    print("  📉 Interpretación OR: DISMINUYE el riesgo de ausentismo alto")
    
    @staticmethod
    def corregir_calculos_or_rr(df, binary_vars, target_bin):
        """Recalcular correctamente OR y RR para todas las variables binarias"""
        print("\n🔧 CORRECCIÓN DE CÁLCULOS OR Y RR:")
        print("-" * 40)
        
        resultados_corregidos = []
        
        for var in binary_vars:
            if var not in df.columns:
                continue
                
            print(f"\n📊 Recalculando {var}:")
            
            try:
                # Tabla de contingencia sin totales
                tabla = pd.crosstab(df[var], df[target_bin])
                
                if tabla.shape != (2, 2):
                    print(f"  ⚠ Tabla no es 2x2: {tabla.shape}")
                    continue
                
                # Extraer valores asegurando el orden correcto
                # Grupo 0 (var=0), Grupo 1 (var=1)
                # Columna 0 (target=0), Columna 1 (target=1)
                a = tabla.iloc[0, 0]  # var=0, target=0
                b = tabla.iloc[0, 1]  # var=0, target=1  
                c = tabla.iloc[1, 0]  # var=1, target=0
                d = tabla.iloc[1, 1]  # var=1, target=1
                
                print(f"  - Tabla: [{a}, {b}; {c}, {d}]")
                
                # CÁLCULOS CORRECTOS:
                
                # 1. Risk Ratio (Ratio de Riesgos)
                riesgo_grupo1 = d / (c + d) if (c + d) > 0 else 0
                riesgo_grupo0 = b / (a + b) if (a + b) > 0 else 0
                risk_ratio = riesgo_grupo1 / riesgo_grupo0 if riesgo_grupo0 > 0 else np.nan
                
                # 2. Odds Ratio
                odds_grupo1 = d / c if c > 0 else np.nan
                odds_grupo0 = b / a if a > 0 else np.nan
                odds_ratio = odds_grupo1 / odds_grupo0 if not np.isnan(odds_grupo1) and not np.isnan(odds_grupo0) else np.nan
                
                # 3. Test exacto de Fisher
                try:
                    odds_ratio_fisher, fisher_p = fisher_exact(tabla)
                except:
                    odds_ratio_fisher, fisher_p = np.nan, np.nan
                
                print(f"  ✓ Risk Ratio corregido: {risk_ratio:.4f}")
                print(f"  ✓ Odds Ratio corregido: {odds_ratio:.4f}")
                print(f"  ✓ Odds Ratio Fisher: {odds_ratio_fisher:.4f}")
                print(f"  ✓ Riesgo Grupo 0: {riesgo_grupo0:.4f}")
                print(f"  ✓ Riesgo Grupo 1: {riesgo_grupo1:.4f}")
                
                resultados_corregidos.append({
                    'Variable': var,
                    'Risk_Ratio_Corregido': risk_ratio,
                    'Odds_Ratio_Corregido': odds_ratio,
                    'Odds_Ratio_Fisher': odds_ratio_fisher,
                    'Fisher_p': fisher_p,
                    'Riesgo_Grupo0': riesgo_grupo0,
                    'Riesgo_Grupo1': riesgo_grupo1,
                    'Diferencia_Riesgo': riesgo_grupo1 - riesgo_grupo0
                })
                
            except Exception as e:
                print(f"  ❌ Error calculando {var}: {e}")
        
        return pd.DataFrame(resultados_corregidos)
    
    def analizar_reason_absence_numeric(self, df):
        """Análisis detallado de Reason_absence_numeric"""
        print("\n🎯 ANÁLISIS DETALLADO DE REASON_ABSENCE_NUMERIC:")
        print("-" * 50)
        
        if 'Reason_absence_numeric' not in df.columns:
            print("❌ Reason_absence_numeric no encontrada")
            return
        
        # Distribución de valores
        print("📊 DISTRIBUCIÓN DE VALORES:")
        print(df['Reason_absence_numeric'].value_counts().sort_index())
        
        # Relación con ausentismo
        print("\n📈 AUSENTISMO POR RAZÓN:")
        ausentismo_por_razon = df.groupby('Reason_absence_numeric')['Absenteeism_hours'].agg([
            'count', 'mean', 'median', 'std', 'min', 'max'
        ]).round(2)
        print(ausentismo_por_razon)
        
        # Correlación detallada
        corr_spearman = df['Reason_absence_numeric'].corr(df['Absenteeism_hours'], method='spearman')
        corr_pearson = df['Reason_absence_numeric'].corr(df['Absenteeism_hours'], method='pearson')
        
        print(f"\n🔗 CORRELACIONES:")
        print(f"  - Spearman: {corr_spearman:.4f}")
        print(f"  - Pearson: {corr_pearson:.4f}")
        
        # Visualización de la relación
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df, x='Reason_absence_numeric', y='Absenteeism_hours')
        plt.title('Distribución de Ausentismo por Reason_absence_numeric')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Asegurar que el directorio existe antes de guardar
        os.makedirs(self.output_path, exist_ok=True)
        plt.savefig(f'{self.output_path}/reason_absence_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✓ Gráfico de análisis guardado: reason_absence_analysis.png")

# Ejecutar verificaciones - PASAR output_path al constructor
corrector = MetricasCorrector(config.OUTPUT_PATH)

# 1. Validar Disciplinary_failure
corrector.verificar_codificacion_disciplinary_failure(df_clean)

# 2. Corregir cálculos OR y RR
df_or_rr_corregidos = corrector.corregir_calculos_or_rr(df_clean, VARIABLES_BINARIAS, TARGET_BINARIO)

# 3. Analizar Reason_absence_numeric
corrector.analizar_reason_absence_numeric(df_clean)


FASE 7.5: VERIFICACIÓN Y CORRECCIÓN DE MÉTRICAS BINARIAS
📁 Directorio de salida verificado: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3

🔍 VALIDACIÓN MANUAL DE DISCIPLINARY_FAILURE:
--------------------------------------------------
📊 DISTRIBUCIÓN DE DISCIPLINARY_FAILURE:
Disciplinary_failure
0    764
1     42
Name: count, dtype: int64
Proporciones: Disciplinary_failure
0    0.947891
1    0.052109
Name: proportion, dtype: float64

📈 AUSENTISMO POR DISCIPLINARY_FAILURE:
                      count  mean  median    std  min    max
Disciplinary_failure                                        
0                       764  7.66     4.0  13.93  0.0  120.0
1                        42  0.00     0.0   0.00  0.0    0.0

🎯 TABLA DE CONTINGENCIA COMPLETA:
Ausentismo_Alto         0   1  Total
Disciplinary_failure                
0                     690  74    764
1                      42   0

7. 6: ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD

In [29]:
print("\n" + "="*80)
print("FASE 7.6: ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD")
print("="*80)

print("\n" + "="*80)
print("ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD")
print("="*80)

class AnalisisDistribuciones:
    """Clase para analizar distribuciones y multicolinealidad"""
    
    @staticmethod
    def analizar_distribuciones_variables_significativas(df, variables):
        """Analizar distribuciones de variables significativas"""
        print("\n📊 ANÁLISIS DE DISTRIBUCIONES:")
        print("-" * 40)
        
        variables_analizar = [v for v in variables if v in df.columns]
        
        for var in variables_analizar:
            print(f"\n🔍 {var}:")
            data = df[var].dropna()
            
            # Estadísticas básicas
            print(f"  - n: {len(data)}")
            print(f"  - Media: {data.mean():.3f}")
            print(f"  - Mediana: {data.median():.3f}")
            print(f"  - Std: {data.std():.3f}")
            print(f"  - Mín: {data.min():.3f}")
            print(f"  - Máx: {data.max():.3f}")
            print(f"  - Asimetría: {stats.skew(data):.3f}")
            print(f"  - Curtosis: {stats.kurtosis(data):.3f}")
            
            # Test de normalidad
            if len(data) > 3:
                stat, p_valor = shapiro(data)
                print(f"  - Shapiro-Wilk: p = {p_valor:.6f} {'(Normal)' if p_valor > 0.05 else '(No normal)'}")
            
            # Sugerencia de transformación
            asimetria = abs(stats.skew(data))
            if asimetria > 1:
                print(f"  💡 Sugerencia: Transformación fuerte recomendada (asimetría: {asimetria:.3f})")
            elif asimetria > 0.5:
                print(f"  💡 Sugerencia: Transformación moderada considerada (asimetría: {asimetria:.3f})")
            else:
                print(f"  💡 Distribución relativamente simétrica (asimetría: {asimetria:.3f})")
    
    @staticmethod
    def aplicar_transformaciones(df, variables):
        """Aplicar y evaluar transformaciones"""
        print("\n🔄 EVALUACIÓN DE TRANSFORMACIONES:")
        print("-" * 40)
        
        resultados_transformaciones = []
        
        for var in variables:
            if var not in df.columns:
                continue
                
            data_original = df[var].dropna()
            if len(data_original) == 0:
                continue
            
            # Solo aplicar a variables continuas no binarias
            if data_original.nunique() > 2 and pd.api.types.is_numeric_dtype(data_original):
                print(f"\n📈 {var}:")
                
                # Transformaciones
                transformaciones = {
                    'Original': data_original,
                    'Log': np.log1p(data_original) if (data_original >= 0).all() else None,
                    'Raíz Cuadrada': np.sqrt(data_original) if (data_original >= 0).all() else None,
                    'Box-Cox': None
                }
                
                # Intentar Box-Cox
                try:
                    if (data_original > 0).all():
                        data_boxcox, _ = stats.boxcox(data_original)
                        transformaciones['Box-Cox'] = data_boxcox
                except:
                    pass
                
                for nombre, data_trans in transformaciones.items():
                    if data_trans is not None:
                        asimetria = stats.skew(data_trans)
                        resultados_transformaciones.append({
                            'Variable': var,
                            'Transformación': nombre,
                            'Asimetría': asimetria,
                            'Mejora': abs(asimetria) < abs(stats.skew(data_original))
                        })
                        print(f"  - {nombre}: asimetría = {asimetria:.3f}")
        
        return pd.DataFrame(resultados_transformaciones)
    
    @staticmethod
    def analizar_multicolinealidad(df, variables):
        """Analizar multicolinealidad entre variables predictoras"""
        print("\n🔗 ANÁLISIS DE MULTICOLINEALIDAD:")
        print("-" * 40)
        
        # Filtrar variables numéricas
        vars_numericas = [v for v in variables if v in df.columns and pd.api.types.is_numeric_dtype(df[v])]
        
        if len(vars_numericas) < 2:
            print("❌ No hay suficientes variables numéricas para analizar multicolinealidad")
            return
        
        # Matriz de correlación
        corr_matrix = df[vars_numericas].corr(method='spearman')
        
        # Encontrar correlaciones altas
        correlaciones_altas = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = abs(corr_matrix.iloc[i, j])
                if corr_val > 0.7:
                    correlaciones_altas.append({
                        'Variable1': corr_matrix.columns[i],
                        'Variable2': corr_matrix.columns[j],
                        'Correlación': corr_val
                    })
        
        print("📊 CORRELACIONES ALTAS (> 0.7):")
        if correlaciones_altas:
            for corr in correlaciones_altas:
                print(f"  ⚠ {corr['Variable1']} - {corr['Variable2']}: {corr['Correlación']:.3f}")
        else:
            print("  ✅ No hay correlaciones altas problemáticas")
        
        # Heatmap de correlaciones
        plt.figure(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                   square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
        plt.title('MATRIZ DE CORRELACIÓN - MULTICOLINEALIDAD', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(f'{config.OUTPUT_PATH}/multicolinealidad_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✓ Heatmap de multicolinealidad guardado")
        
        return correlaciones_altas

# Variables significativas para análisis
VARIABLES_SIGNIFICATIVAS = [
    'Reason_absence_numeric', 'Disciplinary_failure', 'Social_drinker', 
    'Son', 'Transportation_expense'
]

# Ejecutar análisis
analizador = AnalisisDistribuciones()

# 1. Analizar distribuciones
analizador.analizar_distribuciones_variables_significativas(df_clean, VARIABLES_SIGNIFICATIVAS)

# 2. Evaluar transformaciones
df_transformaciones = analizador.aplicar_transformaciones(df_clean, VARIABLES_SIGNIFICATIVAS)

# 3. Analizar multicolinealidad
correlaciones_problematicas = analizador.analizar_multicolinealidad(df_clean, VARIABLES_SIGNIFICATIVAS)


FASE 7.6: ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD

ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD

📊 ANÁLISIS DE DISTRIBUCIONES:
----------------------------------------

🔍 Reason_absence_numeric:
  - n: 806
  - Media: 18.873
  - Mediana: 23.000
  - Std: 8.408
  - Mín: 0.000
  - Máx: 28.000
  - Asimetría: -0.834
  - Curtosis: -0.382
  - Shapiro-Wilk: p = 0.000000 (No normal)
  💡 Sugerencia: Transformación moderada considerada (asimetría: 0.834)

🔍 Disciplinary_failure:
  - n: 806
  - Media: 0.052
  - Mediana: 0.000
  - Std: 0.222
  - Mín: 0.000
  - Máx: 1.000
  - Asimetría: 4.031
  - Curtosis: 14.245
  - Shapiro-Wilk: p = 0.000000 (No normal)
  💡 Sugerencia: Transformación fuerte recomendada (asimetría: 4.031)

🔍 Social_drinker:
  - n: 806
  - Media: 0.562
  - Mediana: 1.000
  - Std: 0.496
  - Mín: 0.000
  - Máx: 1.000
  - Asimetría: -0.250
  - Curtosis: -1.937
  - Shapiro-Wilk: p = 0.000000 (No normal)
  💡 Distribución relativamente simétrica (asimetría: 0.250)

🔍 Son:
  - n: 

7. 7: RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES

In [30]:
print("\n" + "="*80)
print("FASE 7.7: RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES")
print("="*80)

print("\n" + "="*80)
print("RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES")
print("="*80)

class ResumenCorrecciones:
    """Resumen final de todas las correcciones y recomendaciones"""
    
    @staticmethod
    def generar_resumen_completo(df_or_rr_corregidos, correlaciones_problematicas, df_transformaciones):
        """Generar resumen completo de correcciones"""
        
        print("\n🎯 RESUMEN EJECUTIVO DE CORRECCIONES:")
        print("="*50)
        
        # 1. Correcciones OR/RR
        print("\n1. ✅ CORRECCIONES OR/RR APLICADAS:")
        if not df_or_rr_corregidos.empty:
            for _, row in df_or_rr_corregidos.iterrows():
                print(f"   📊 {row['Variable']}:")
                print(f"      - Risk Ratio: {row['Risk_Ratio_Corregido']:.3f}")
                print(f"      - Odds Ratio: {row['Odds_Ratio_Corregido']:.3f}")
                
                # Interpretación corregida
                if row['Risk_Ratio_Corregido'] > 1:
                    print(f"      📈 AUMENTA el riesgo de ausentismo alto")
                else:
                    print(f"      📉 DISMINUYE el riesgo de ausentismo alto")
        
        # 2. Multicolinealidad
        print("\n2. 🔗 ANÁLISIS DE MULTICOLINEALIDAD:")
        if correlaciones_problematicas:
            print("   ⚠ CORRELACIONES PROBLEMÁTICAS DETECTADAS:")
            for corr in correlaciones_problematicas:
                print(f"      - {corr['Variable1']} ↔ {corr['Variable2']}: {corr['Correlación']:.3f}")
            print("   💡 RECOMENDACIÓN: Considerar eliminar una variable de cada par correlacionado")
        else:
            print("   ✅ No se detectaron problemas de multicolinealidad")
        
        # 3. Transformaciones
        print("\n3. 🔄 RECOMENDACIONES DE TRANSFORMACIONES:")
        if not df_transformaciones.empty:
            # Encontrar mejor transformación para cada variable
            variables_unicas = df_transformaciones['Variable'].unique()
            for var in variables_unicas:
                data_var = df_transformaciones[df_transformaciones['Variable'] == var]
                mejor_transform = data_var.loc[data_var['Asimetría'].abs().idxmin()]
                print(f"   📈 {var}:")
                print(f"      - Mejor transformación: {mejor_transform['Transformación']}")
                print(f"      - Asimetría resultante: {mejor_transform['Asimetría']:.3f}")
        
        # 4. Recomendaciones finales para modelo multivariable
        print("\n4. 🎯 RECOMENDACIONES FINALES PARA MODELO MULTIVARIABLE:")
        print("   🥇 VARIABLES PRIORITARIAS (alto impacto):")
        print("      - Reason_absence_numeric (Cohen's d = -0.630)")
        print("      - Disciplinary_failure (efecto significativo pero requiere validación)")
        
        print("\n   🥈 VARIABLES SECUNDARIAS (impacto moderado):")
        print("      - Social_drinker (Cohen's d = 0.374)")
        print("      - Son (Cohen's d = 0.226)")
        
        print("\n   🥉 VARIABLE A EVALUAR (impacto bajo):")
        print("      - Transportation_expense (Cohen's d = 0.032)")
        
        print("\n5. 🚨 ACCIONES CRÍTICAS ANTES DEL MODELO MULTIVARIABLE:")
        print("   ✅ Validar codificación de Disciplinary_failure")
        print("   ✅ Usar OR/RR corregidos en el análisis")
        print("   ✅ Considerar transformaciones para mejorar distribuciones")
        print("   ✅ Evaluar interacciones entre variables personales")
        print("   ✅ Realizar análisis de sensibilidad excluyendo Transportation_expense")

# Ejecutar resumen
resumen = ResumenCorrecciones()
resumen.generar_resumen_completo(df_or_rr_corregidos, correlaciones_problematicas, df_transformaciones)

print(f"\n📁 Todos los análisis detallados guardados en: {config.OUTPUT_PATH}")
print("✅ CORRECCIONES COMPLETADAS - Listo para análisis multivariable")


FASE 7.7: RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES

RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES

🎯 RESUMEN EJECUTIVO DE CORRECCIONES:

1. ✅ CORRECCIONES OR/RR APLICADAS:
   📊 Disciplinary_failure:
      - Risk Ratio: 0.000
      - Odds Ratio: 0.000
      📉 DISMINUYE el riesgo de ausentismo alto
   📊 Social_drinker:
      - Risk Ratio: 2.104
      - Odds Ratio: 2.253
      📈 AUMENTA el riesgo de ausentismo alto
   📊 Social_smoker:
      - Risk Ratio: 1.023
      - Odds Ratio: 1.025
      📈 AUMENTA el riesgo de ausentismo alto

2. 🔗 ANÁLISIS DE MULTICOLINEALIDAD:
   ✅ No se detectaron problemas de multicolinealidad

3. 🔄 RECOMENDACIONES DE TRANSFORMACIONES:
   📈 Reason_absence_numeric:
      - Mejor transformación: Original
      - Asimetría resultante: -0.834
   📈 Son:
      - Mejor transformación: Raíz Cuadrada
      - Asimetría resultante: -0.011
   📈 Transportation_expense:
      - Mejor transformación: Box-Cox
      - Asimetría resultante: -0.022

4. 🎯 RECOMENDACIONES FI

8. Visualizaciones (Modularizado)

In [31]:
print("\n" + "="*80)
print("FASE 8: GENERACIÓN DE VISUALIZACIONES")
print("="*80)

class VisualizacionGenerator:
    def __init__(self, output_path):
        self.output_path = output_path
        import os
        os.makedirs(output_path, exist_ok=True)
        print(f"📁 Directorio de salida: {output_path}")
    
    def generar_heatmap_corregido(self, df, variables_personales, variables_laborales, target):
        """Heatmap corregido que solo usa variables numéricas verificadas"""
        print("📊 Generando heatmap de correlaciones CORREGIDO...")
        
        # Filtrar solo variables numéricas verificadas
        vars_numericas_personales = [var for var in variables_personales 
                                   if var in df.columns and pd.api.types.is_numeric_dtype(df[var])]
        vars_numericas_laborales = [var for var in variables_laborales 
                                  if var in df.columns and pd.api.types.is_numeric_dtype(df[var])]
        
        corr_vars = vars_numericas_personales + vars_numericas_laborales + [target]
        
        print(f"  - Variables personales numéricas: {len(vars_numericas_personales)}")
        print(f"  - Variables laborales numéricas: {len(vars_numericas_laborales)}")
        print(f"  - Total variables para heatmap: {len(corr_vars)}")
        
        if len(corr_vars) > 1:
            try:
                # Calcular matriz de correlación
                corr_matrix = df[corr_vars].corr(method='spearman')
                
                # Crear heatmap
                mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
                plt.figure(figsize=(16, 14))
                sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                           square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
                plt.title('MATRIZ DE CORRELACIÓN SPEARMAN - AUSENTISMO LABORAL\n(Variables Personales y Laborales Numéricas)', 
                         fontsize=16, fontweight='bold', pad=20)
                plt.tight_layout()
                plt.savefig(f'{self.output_path}/1_heatmap_correlaciones.png', dpi=300, bbox_inches='tight')
                plt.close()
                print("✓ Heatmap de correlaciones guardado")
                
            except Exception as e:
                print(f"✗ Error generando heatmap: {e}")
                self._generar_heatmap_simplificado(df, corr_vars)
        else:
            print("⚠ No hay suficientes variables numéricas para generar heatmap")
    
    def _generar_heatmap_simplificado(self, df, corr_vars):
        """Heatmap simplificado de respaldo"""
        try:
            print("  🔧 Generando heatmap simplificado...")
            corr_matrix = df[corr_vars].corr(method='spearman')
            plt.figure(figsize=(12, 10))
            sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
            plt.title('Correlación Spearman - Versión Simplificada', fontsize=14)
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/1_heatmap_simplificado.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Heatmap simplificado guardado")
        except Exception as e:
            print(f"✗ Error en heatmap simplificado: {e}")
    
    def generar_importancia_variables(self, df_results):
        print("📊 Generando gráfico de importancia...")
        if df_results.empty:
            print("⚠ No hay datos para generar gráfico de importancia")
            return
        
        try:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
            
            # Gráfico de correlación
            corr_data = df_results.sort_values('abs_corr', ascending=True)
            colors = ['red' if sig else 'gray' for sig in corr_data['Significativa_Spearman']]
            ax1.barh(corr_data['Variable'], corr_data['abs_corr'], color=colors, alpha=0.7)
            ax1.set_xlabel('Correlación Absoluta (Spearman)')
            ax1.set_title('IMPORTANCIA POR CORRELACIÓN', fontweight='bold')
            ax1.grid(axis='x', alpha=0.3)
            
            # Gráfico de tamaño del efecto
            effect_data = df_results.sort_values('abs_effect', ascending=True)
            colors_effect = ['red' if sig else 'gray' for sig in effect_data['Significativa_Medias']]
            ax2.barh(effect_data['Variable'], effect_data['abs_effect'], color=colors_effect, alpha=0.7)
            ax2.set_xlabel('Tamaño del Efecto Absoluto (Cohen d)')
            ax2.set_title('IMPORTANCIA POR TAMAÑO DEL EFECTO', fontweight='bold')
            ax2.grid(axis='x', alpha=0.3)
            
            plt.suptitle('COMPARACIÓN DE IMPORTANCIA DE VARIABLES', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/2_importancia_variables.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Gráfico de importancia de variables guardado")
        except Exception as e:
            print(f"✗ Error generando gráfico de importancia: {e}")
    
    def generar_scatter_variables_significativas(self, df, df_results, target, max_variables=6):
        print("📊 Generando scatter plots de variables significativas...")
        if df_results.empty:
            print("⚠ No hay resultados para generar scatter plots")
            return
        
        try:
            # Filtrar solo variables significativas
            sig_vars_df = df_results[df_results['Significativa_Spearman']]
            if sig_vars_df.empty:
                print("⚠ No hay variables significativas para scatter plots")
                return
            
            sig_vars = sig_vars_df.nlargest(max_variables, 'abs_corr')['Variable'].tolist()
            
            # Filtrar solo variables numéricas
            sig_vars_numericas = [var for var in sig_vars 
                                if var in df.columns and pd.api.types.is_numeric_dtype(df[var])]
            
            n_vars = len(sig_vars_numericas)
            if n_vars == 0:
                print("⚠ No hay variables numéricas significativas para scatter plots")
                return
            
            n_cols = min(3, n_vars)
            n_rows = (n_vars + n_cols - 1) // n_cols
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
            
            if n_rows == 1 and n_cols == 1:
                axes = [axes]
            elif n_rows == 1:
                axes = axes
            else:
                axes = axes.flatten()
            
            for i, var in enumerate(sig_vars_numericas):
                if i < len(axes):
                    sns.regplot(data=df, x=var, y=target, ax=axes[i], 
                               scatter_kws={'alpha':0.6, 's':30}, 
                               line_kws={'color':'red'}, ci=95)
                    stats_row = df_results[df_results['Variable'] == var].iloc[0]
                    corr = stats_row['Correlacion_Spearman']
                    fuerza = stats_row['Fuerza_Spearman']
                    direccion = "Positiva" if corr > 0 else "Negativa"
                    title_text = f'{var}\nρ = {corr:.3f} ({fuerza}, {direccion})'
                    axes[i].set_title(title_text, fontweight='bold', fontsize=10)
                    axes[i].set_xlabel(var)
                    axes[i].set_ylabel('Horas de Ausentismo')
            
            for j in range(len(sig_vars_numericas), len(axes)):
                axes[j].set_visible(False)
            
            plt.suptitle('RELACIÓN DE VARIABLES MÁS SIGNIFICATIVAS CON AUSENTISMO', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/3_scatter_variables_significativas.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Scatter plots de variables significativas guardados")
        except Exception as e:
            print(f"✗ Error generando scatter plots: {e}")
    
    def generar_boxplots_categoricas(self, df, df_results, target):
        print("📊 Generando boxplots de variables categóricas...")
        if df_results.empty:
            print("⚠ No hay resultados categóricos para generar boxplots")
            return
        
        try:
            # Filtrar solo variables categóricas significativas
            sig_cat_vars = df_results[df_results['Significativa_Principal']]['Variable'].tolist()
            if not sig_cat_vars:
                print("⚠ No hay variables categóricas significativas")
                return
            
            n_cat_vars = len(sig_cat_vars)
            n_cols = min(2, n_cat_vars)
            n_rows = (n_cat_vars + n_cols - 1) // n_cols
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
            
            if n_rows == 1 and n_cols == 1:
                axes = [axes]
            elif n_rows == 1:
                axes = axes
            else:
                axes = axes.flatten()
            
            for i, cat_var in enumerate(sig_cat_vars):
                if i < len(axes) and cat_var in df.columns:
                    order = df.groupby(cat_var)[target].median().sort_values(ascending=False).index
                    sns.boxplot(data=df, x=cat_var, y=target, ax=axes[i], order=order)
                    axes[i].set_title(f'Ausentismo por {cat_var}', fontweight='bold')
                    axes[i].tick_params(axis='x', rotation=45)
                    axes[i].set_xlabel(cat_var)
                    axes[i].set_ylabel('Horas de Ausentismo')
            
            for j in range(len(sig_cat_vars), len(axes)):
                axes[j].set_visible(False)
            
            plt.suptitle('DISTRIBUCIÓN DE AUSENTISMO POR VARIABLES CATEGÓRICAS', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/4_boxplots_categoricas.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Boxplots de variables categóricas guardados")
        except Exception as e:
            print(f"✗ Error generando boxplots: {e}")
    
    def generar_grafico_correlaciones_impacto(self, df_results, variables_personales, variables_laborales):
        """Gráfico 5 corregido: muestra solo variables personales y laborales significativas"""
        print("📊 Generando gráfico de correlaciones de impacto...")
        if df_results.empty:
            print("⚠ No hay resultados para generar gráfico de impacto")
            return
        
        try:
            # FILTRAR SOLO VARIABLES SIGNIFICATIVAS que son personales o laborales
            df_significativas = df_results[
                df_results['Significativa_Spearman'] & 
                (df_results['Variable'].isin(variables_personales + variables_laborales))
            ].copy()
            
            if df_significativas.empty:
                print("⚠ No hay variables personales o laborales significativas para el gráfico de impacto")
                return
                
            # Ordenar por correlación absoluta
            df_plot = df_significativas.sort_values('abs_corr', ascending=True)
            
            plt.figure(figsize=(14, 10))
            
            # Asignar colores según tipo de variable
            colors = []
            for _, row in df_plot.iterrows():
                if row['Variable'] in variables_personales:
                    colors.append((0.2, 0.4, 0.8, 0.8))  # Azul para personales
                else:
                    colors.append((0.8, 0.4, 0.2, 0.8))  # Naranja para laborales
            
            # Crear barras
            bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], color=colors, alpha=0.8)
            
            # Añadir valores en las barras
            for i, (value, variable) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['Variable'])):
                if value >= 0:
                    plt.text(value + 0.01, i, f'{value:.3f}', va='center', ha='left', fontweight='bold', fontsize=10)
                else:
                    plt.text(value - 0.01, i, f'{value:.3f}', va='center', ha='right', fontweight='bold', fontsize=10)
            
            plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
            plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=12, fontweight='bold')
            
            # Título indicando que solo muestra variables personales y laborales significativas
            plt.title('IMPACTO DE VARIABLES PERSONALES Y LABORALES EN EL AUSENTISMO\n(Solo variables significativas p < 0.05)', 
                     fontsize=16, fontweight='bold', pad=20)
            
            # Añadir anotaciones de efecto y tipo
            for i, (_, row) in enumerate(df_plot.iterrows()):
                effect_text = "Aumenta ausentismo" if row['Correlacion_Spearman'] > 0 else "Disminuye ausentismo"
                tipo = "PERSONAL" if row['Variable'] in variables_personales else "LABORAL"
                color = 'darkblue' if tipo == 'PERSONAL' else 'darkorange'
                plt.annotate(f"{effect_text} ({tipo})", 
                            xy=(row['Correlacion_Spearman'], i), 
                            xytext=(5, 0), 
                            textcoords='offset points', 
                            ha='left', va='center', 
                            fontsize=9, color=color, fontweight='bold')
            
            # Añadir leyenda
            from matplotlib.patches import Patch
            legend_elements = [
                Patch(facecolor=(0.2, 0.4, 0.8, 0.8), label='Variables Personales'),
                Patch(facecolor=(0.8, 0.4, 0.2, 0.8), label='Variables Laborales')
            ]
            plt.legend(handles=legend_elements, loc='lower right')
            
            plt.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/5_grafico_correlaciones_impacto.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Gráfico de correlaciones de impacto guardado")
        except Exception as e:
            print(f"✗ Error generando gráfico de impacto: {e}")
    
    def generar_todas_visualizaciones(self, df, df_results_numeric, df_results_categorical, target, variables_personales, variables_laborales):
        print("\n🎨 GENERANDO TODAS LAS VISUALIZACIONES...")
        print("-" * 50)
        
        # Heatmap corregido
        self.generar_heatmap_corregido(df, variables_personales, variables_laborales, target)
        
        # Gráficos que usan resultados estadísticos
        self.generar_importancia_variables(df_results_numeric)
        self.generar_scatter_variables_significativas(df, df_results_numeric, target)
        self.generar_boxplots_categoricas(df, df_results_categorical, target)
        self.generar_grafico_correlaciones_impacto(df_results_numeric, variables_personales, variables_laborales)
        
        print("\n✅ TODAS LAS VISUALIZACIONES GENERADAS EXITOSAMENTE!")

# EJECUCIÓN CORREGIDA
try:
    # Asegurar que tenemos la columna Fuerza_Num
    if 'Fuerza_Num' not in df_results_numeric.columns:
        def clasificar_correlacion(corr):
            abs_corr = abs(corr)
            if abs_corr < 0.1: return "Muy débil", 1
            elif abs_corr < 0.3: return "Débil", 2
            elif abs_corr < 0.5: return "Moderada", 3
            elif abs_corr < 0.7: return "Fuerte", 4
            else: return "Muy fuerte", 5
        
        resultados = df_results_numeric['Correlacion_Spearman'].apply(clasificar_correlacion)
        df_temp = pd.DataFrame(resultados.tolist(), columns=['Fuerza_Spearman', 'Fuerza_Num'], index=df_results_numeric.index)
        df_results_numeric = pd.concat([df_results_numeric, df_temp], axis=1)

    print(f"📁 Creando visualizaciones en: {config.OUTPUT_PATH}")
    viz = VisualizacionGenerator(config.OUTPUT_PATH)
    
    # Llamada CORREGIDA - solo pasamos las listas necesarias
    viz.generar_todas_visualizaciones(
        df_clean, 
        df_results_numeric, 
        df_results_categorical, 
        TARGET_CONTINUO,
        VARIABLES_PERSONALES,
        VARIABLES_LABORALES
    )

    print(f"\n📁 Visualizaciones guardadas en: {config.OUTPUT_PATH}")
    print("✓ 1_heatmap_correlaciones.png")
    print("✓ 2_importancia_variables.png") 
    print("✓ 3_scatter_variables_significativas.png")
    print("✓ 4_boxplots_categoricas.png")
    print("✓ 5_grafico_correlaciones_impacto.png")

except Exception as e:
    print(f"❌ Error en visualizaciones: {e}")
    print("Generando visualización mínima de respaldo...")
    try:
        # Respaldo seguro - solo variables numéricas verificadas
        vars_numericas_seguras = [var for var in VARIABLES_PERSONALES + VARIABLES_LABORALES 
                                if var in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean[var])]
        vars_para_respaldo = vars_numericas_seguras[:8] + [TARGET_CONTINUO]
        
        if len(vars_para_respaldo) > 2:
            plt.figure(figsize=(10, 8))
            corr_matrix = df_clean[vars_para_respaldo].corr(method='spearman')
            sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r')
            plt.title('Correlación con Ausentismo - Respaldo')
            plt.tight_layout()
            plt.savefig(f'{config.OUTPUT_PATH}/backup_heatmap.png')
            plt.close()
            print("✓ Visualización de respaldo guardada")
    except Exception as backup_error:
        print(f"✗ Error en visualización de respaldo: {backup_error}")


FASE 8: GENERACIÓN DE VISUALIZACIONES
📁 Creando visualizaciones en: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3
📁 Directorio de salida: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3

🎨 GENERANDO TODAS LAS VISUALIZACIONES...
--------------------------------------------------
📊 Generando heatmap de correlaciones CORREGIDO...
  - Variables personales numéricas: 9
  - Variables laborales numéricas: 6
  - Total variables para heatmap: 16
✓ Heatmap de correlaciones guardado
📊 Generando gráfico de importancia...
✓ Gráfico de importancia de variables guardado
📊 Generando scatter plots de variables significativas...
✓ Scatter plots de variables significativas guardados
📊 Generando boxplots de variables categóricas...
✓ Boxplots de variables categóricas guardados
📊 Generando gráfico de co

In [32]:
print("\n" + "="*80)
print("NUEVO GRÁFICO: MAGNITUD, DIRECCIÓN Y SIGNIFICANCIA")
print("="*80)

def generar_grafico_magnitud_direccion_significancia(df_results, output_path):
    """Generar gráfico mejorado que muestra magnitud, dirección y significancia"""
    print("📊 Generando gráfico de magnitud, dirección y significancia...")
    
    if df_results.empty:
        print("⚠ No hay resultados para generar el gráfico")
        return
    
    try:
        # Filtrar solo variables significativas
        df_significativas = df_results[df_results['Significativa_Spearman']].copy()
        
        if df_significativas.empty:
            print("⚠ No hay variables significativas para el gráfico")
            return
        
        # Ordenar por correlación absoluta (magnitud)
        df_plot = df_significativas.sort_values('abs_corr', ascending=True)
        
        # Crear figura
        plt.figure(figsize=(14, 10))
        
        # Configurar colores y estilos
        colors = []
        edgecolors = []
        linewidths = []
        
        for _, row in df_plot.iterrows():
            # Color por dirección
            if row['Correlacion_Spearman'] > 0:
                color = (0.8, 0.2, 0.2, 0.8)  # Rojo para positivo
            else:
                color = (0.2, 0.4, 0.8, 0.8)  # Azul para negativo
            
            # Borde más grueso para mayor significancia (p < 0.01)
            if row['p_value_Spearman'] < 0.01:
                edgecolor = 'black'
                linewidth = 2.5
            elif row['p_value_Spearman'] < 0.05:
                edgecolor = 'darkgray'
                linewidth = 1.5
            else:
                edgecolor = 'lightgray'
                linewidth = 1.0
            
            colors.append(color)
            edgecolors.append(edgecolor)
            linewidths.append(linewidth)
        
        # Crear barras horizontales
        bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], 
                       color=colors, alpha=0.8, edgecolor=edgecolors, linewidth=linewidths)
        
        # Añadir línea vertical en 0
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=1)
        
        # Añadir valores en las barras
        for i, (value, p_valor) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['p_value_Spearman'])):
            # Determinar símbolo de significancia
            if p_valor < 0.001:
                simbolo = '***'
            elif p_valor < 0.01:
                simbolo = '**'
            elif p_valor < 0.05:
                simbolo = '*'
            else:
                simbolo = ''
            
            # Posicionar texto
            if value >= 0:
                x_pos = value + 0.01
                ha = 'left'
            else:
                x_pos = value - 0.01
                ha = 'right'
            
            # Texto con valor y significancia
            texto = f'{value:.3f}{simbolo}'
            plt.text(x_pos, i, texto, va='center', ha=ha, fontweight='bold', 
                    fontsize=10, bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
        
        # Configurar ejes y título
        plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=12, fontweight='bold')
        plt.ylabel('Variables', fontsize=12, fontweight='bold')
        
        # Título informativo
        plt.title('IMPACTO EN AUSENTISMO LABORAL\n(Magnitud, Dirección y Significancia Estadística)', 
                 fontsize=16, fontweight='bold', pad=20)
        
        # Añadir leyenda de significancia
        legend_elements = [
            plt.Rectangle((0,0), 1, 1, facecolor='red', alpha=0.7, edgecolor='black', label='Aumenta Ausentismo'),
            plt.Rectangle((0,0), 1, 1, facecolor='blue', alpha=0.7, edgecolor='black', label='Disminuye Ausentismo'),
            plt.Line2D([0], [0], color='black', linewidth=2.5, label='p < 0.01'),
            plt.Line2D([0], [0], color='darkgray', linewidth=1.5, label='p < 0.05')
        ]
        
        plt.legend(handles=legend_elements, loc='lower right', framealpha=0.9)
        
        # Añadir grid para mejor lectura
        plt.grid(axis='x', alpha=0.3, linestyle='--')
        plt.grid(axis='y', alpha=0.1, linestyle='--')
        
        # Ajustar límites del eje x para mejor visualización
        x_margin = max(abs(df_plot['Correlacion_Spearman'])) * 0.15
        plt.xlim(min(df_plot['Correlacion_Spearman']) - x_margin, 
                max(df_plot['Correlacion_Spearman']) + x_margin)
        
        # Mejorar estética general
        plt.tight_layout()
        
        # Guardar gráfico
        plt.savefig(f'{output_path}/6_grafico_magnitud_direccion_significancia.png', 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print("✓ Gráfico de magnitud, dirección y significancia guardado")
        
        # Información adicional sobre las variables
        print(f"\n📋 RESUMEN ESTADÍSTICO DEL GRÁFICO:")
        print(f"  - Variables mostradas: {len(df_plot)}")
        print(f"  - Correlación más fuerte: {df_plot['Correlacion_Spearman'].abs().max():.3f}")
        print(f"  - Correlación más débil: {df_plot['Correlacion_Spearman'].abs().min():.3f}")
        print(f"  - Variables que aumentan ausentismo: {len(df_plot[df_plot['Correlacion_Spearman'] > 0])}")
        print(f"  - Variables que disminuyen ausentismo: {len(df_plot[df_plot['Correlacion_Spearman'] < 0])}")
        
    except Exception as e:
        print(f"✗ Error generando gráfico: {e}")

# Ejecutar la función
generar_grafico_magnitud_direccion_significancia(df_results_numeric, config.OUTPUT_PATH)

print(f"\n📁 Gráfico guardado en: {config.OUTPUT_PATH}/6_grafico_magnitud_direccion_significancia.png")


NUEVO GRÁFICO: MAGNITUD, DIRECCIÓN Y SIGNIFICANCIA
📊 Generando gráfico de magnitud, dirección y significancia...
✓ Gráfico de magnitud, dirección y significancia guardado

📋 RESUMEN ESTADÍSTICO DEL GRÁFICO:
  - Variables mostradas: 4
  - Correlación más fuerte: 0.390
  - Correlación más débil: 0.121
  - Variables que aumentan ausentismo: 3
  - Variables que disminuyen ausentismo: 1

📁 Gráfico guardado en: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3/6_grafico_magnitud_direccion_significancia.png


In [33]:
print("\n" + "="*80)
print("NUEVO GRÁFICO: MAGNITUD, DIRECCIÓN Y SIGNIFICANCIA (LETRAS GRANDES)")
print("="*80)

def generar_grafico_magnitud_direccion_significancia_grande(df_results, output_path):
    """Generar gráfico mejorado con letras grandes para mejor legibilidad"""
    print("📊 Generando gráfico con letras grandes para mejor visualización...")
    
    if df_results.empty:
        print("⚠ No hay resultados para generar el gráfico")
        return
    
    try:
        # Filtrar solo variables significativas
        df_significativas = df_results[df_results['Significativa_Spearman']].copy()
        
        if df_significativas.empty:
            print("⚠ No hay variables significativas para el gráfico")
            return
        
        # Ordenar por correlación absoluta (magnitud)
        df_plot = df_significativas.sort_values('abs_corr', ascending=True)
        
        # Crear figura MÁS GRANDE para mejor legibilidad
        plt.figure(figsize=(16, 12))
        
        # Configurar colores y estilos
        colors = []
        edgecolors = []
        linewidths = []
        
        for _, row in df_plot.iterrows():
            # Color por dirección
            if row['Correlacion_Spearman'] > 0:
                color = (0.8, 0.2, 0.2, 0.8)  # Rojo para positivo
            else:
                color = (0.2, 0.4, 0.8, 0.8)  # Azul para negativo
            
            # Borde más grueso para mayor significancia (p < 0.01)
            if row['p_value_Spearman'] < 0.01:
                edgecolor = 'black'
                linewidth = 3.0
            elif row['p_value_Spearman'] < 0.05:
                edgecolor = 'darkgray'
                linewidth = 2.0
            else:
                edgecolor = 'lightgray'
                linewidth = 1.5
            
            colors.append(color)
            edgecolors.append(edgecolor)
            linewidths.append(linewidth)
        
        # Crear barras horizontales MÁS GRUESAS
        bar_height = 0.8
        bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], 
                       height=bar_height, color=colors, alpha=0.8, 
                       edgecolor=edgecolors, linewidth=linewidths)
        
        # Añadir línea vertical en 0
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=2)
        
        # Añadir valores en las barras CON LETRAS MÁS GRANDES
        for i, (value, p_valor) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['p_value_Spearman'])):
            # Determinar símbolo de significancia
            if p_valor < 0.001:
                simbolo = '***'
            elif p_valor < 0.01:
                simbolo = '**'
            elif p_valor < 0.05:
                simbolo = '*'
            else:
                simbolo = ''
            
            # Posicionar texto
            if value >= 0:
                x_pos = value + 0.015
                ha = 'left'
            else:
                x_pos = value - 0.015
                ha = 'right'
            
            # Texto con valor y significancia - LETRAS MUCHO MÁS GRANDES
            texto = f'{value:.3f}{simbolo}'
            plt.text(x_pos, i, texto, va='center', ha=ha, fontweight='bold', 
                    fontsize=16,  # Aumentado de 10 a 16
                    bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.9, edgecolor='gray'))
        
        # Configurar ejes y título CON LETRAS MÁS GRANDES
        plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=18, fontweight='bold')  # Aumentado de 12 a 18
        plt.ylabel('Variables', fontsize=18, fontweight='bold')  # Aumentado de 12 a 18
        
        # Aumentar tamaño de letra de las variables en el eje Y
        plt.yticks(fontsize=16)  # Aumentado significativamente
        plt.xticks(fontsize=14)  # Aumentado para los números del eje X
        
        # Título informativo CON LETRA MÁS GRANDE
        plt.title('IMPACTO EN AUSENTISMO LABORAL\n(Magnitud, Dirección y Significancia Estadística)', 
                 fontsize=20, fontweight='bold', pad=30)  # Aumentado de 16 a 20
        
        # Añadir leyenda de significancia CON LETRAS MÁS GRANDES
        legend_elements = [
            plt.Rectangle((0,0), 1, 1, facecolor='red', alpha=0.7, edgecolor='black', label='Aumenta Ausentismo'),
            plt.Rectangle((0,0), 1, 1, facecolor='blue', alpha=0.7, edgecolor='black', label='Disminuye Ausentismo'),
            plt.Line2D([0], [0], color='black', linewidth=3.0, label='p < 0.01'),
            plt.Line2D([0], [0], color='darkgray', linewidth=2.0, label='p < 0.05')
        ]
        
        plt.legend(handles=legend_elements, loc='lower right', framealpha=0.9, fontsize=14)  # Aumentado el tamaño de letra de la leyenda
        
        # Añadir grid para mejor lectura
        plt.grid(axis='x', alpha=0.3, linestyle='--')
        plt.grid(axis='y', alpha=0.1, linestyle='--')
        
        # Ajustar límites del eje x para mejor visualización
        x_margin = max(abs(df_plot['Correlacion_Spearman'])) * 0.15
        plt.xlim(min(df_plot['Correlacion_Spearman']) - x_margin, 
                max(df_plot['Correlacion_Spearman']) + x_margin)
        
        # Ajustar márgenes para que quepan las letras grandes
        plt.subplots_adjust(left=0.3, right=0.95, top=0.9, bottom=0.1)
        
        # Mejorar estética general
        plt.tight_layout()
        
        # Guardar gráfico con alta resolución
        plt.savefig(f'{output_path}/6_grafico_magnitud_direccion_significancia_GRANDE.png', 
                   dpi=350, bbox_inches='tight', facecolor='white')  # Aumentado DPI a 350
        plt.close()
        
        print("✓ Gráfico con letras grandes guardado")
        
        # Información adicional sobre las variables
        print(f"\n📋 RESUMEN ESTADÍSTICO DEL GRÁFICO:")
        print(f"  - Variables mostradas: {len(df_plot)}")
        print(f"  - Correlación más fuerte: {df_plot['Correlacion_Spearman'].abs().max():.3f}")
        print(f"  - Correlación más débil: {df_plot['Correlacion_Spearman'].abs().min():.3f}")
        print(f"  - Variables que aumentan ausentismo: {len(df_plot[df_plot['Correlacion_Spearman'] > 0])}")
        print(f"  - Variables que disminuyen ausentismo: {len(df_plot[df_plot['Correlacion_Spearman'] < 0])}")
        
        # Mostrar detalles de cada variable
        print(f"\n🔍 DETALLES POR VARIABLE:")
        for _, row in df_plot.iterrows():
            direccion = "AUMENTA" if row['Correlacion_Spearman'] > 0 else "DISMINUYE"
            print(f"  • {row['Variable']}: {direccion} (ρ={row['Correlacion_Spearman']:.3f}, p={row['p_value_Spearman']:.4f})")
        
    except Exception as e:
        print(f"✗ Error generando gráfico: {e}")

# También creamos una versión EXTRA GRANDE para presentaciones
def generar_grafico_extra_grande(df_results, output_path):
    """Versión extra grande para presentaciones o informes"""
    print("\n📊 Generando versión EXTRA GRANDE para presentaciones...")
    
    if df_results.empty:
        return
    
    try:
        df_significativas = df_results[df_results['Significativa_Spearman']].copy()
        if df_significativas.empty:
            return
        
        df_plot = df_significativas.sort_values('abs_corr', ascending=True)
        
        # Figura EXTRA GRANDE
        plt.figure(figsize=(20, 14))
        
        colors = []
        for _, row in df_plot.iterrows():
            colors.append((0.8, 0.2, 0.2, 0.8) if row['Correlacion_Spearman'] > 0 else (0.2, 0.4, 0.8, 0.8))
        
        # Barras
        bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], 
                       height=0.9, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
        
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=3)
        
        # Valores con letras EXTRA GRANDES
        for i, (value, p_valor) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['p_value_Spearman'])):
            simbolo = '***' if p_valor < 0.001 else '**' if p_valor < 0.01 else '*' if p_valor < 0.05 else ''
            
            if value >= 0:
                x_pos = value + 0.02
                ha = 'left'
            else:
                x_pos = value - 0.02
                ha = 'right'
            
            texto = f'{value:.3f}{simbolo}'
            plt.text(x_pos, i, texto, va='center', ha=ha, fontweight='bold', 
                    fontsize=20,  # MUY GRANDE
                    bbox=dict(boxstyle="round,pad=0.8", facecolor='white', alpha=0.9))
        
        # Etiquetas de ejes EXTRA GRANDES
        plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=22, fontweight='bold')
        plt.ylabel('Variables', fontsize=22, fontweight='bold')
        plt.yticks(fontsize=18)
        plt.xticks(fontsize=16)
        
        # Título EXTRA GRANDE
        plt.title('IMPACTO EN AUSENTISMO LABORAL\n(Magnitud, Dirección y Significancia Estadística)', 
                 fontsize=24, fontweight='bold', pad=40)
        
        plt.grid(axis='x', alpha=0.3, linestyle='--')
        
        x_margin = max(abs(df_plot['Correlacion_Spearman'])) * 0.15
        plt.xlim(min(df_plot['Correlacion_Spearman']) - x_margin, 
                max(df_plot['Correlacion_Spearman']) + x_margin)
        
        plt.subplots_adjust(left=0.35, right=0.95, top=0.92, bottom=0.08)
        
        plt.savefig(f'{output_path}/7_grafico_EXTRA_GRANDE.png', 
                   dpi=400, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print("✓ Versión EXTRA GRANDE guardada")
        
    except Exception as e:
        print(f"✗ Error en versión extra grande: {e}")

# Ejecutar ambas versiones
generar_grafico_magnitud_direccion_significancia_grande(df_results_numeric, config.OUTPUT_PATH)
generar_grafico_extra_grande(df_results_numeric, config.OUTPUT_PATH)

print(f"\n📁 Gráficos guardados en:")
print(f"  • {config.OUTPUT_PATH}/6_grafico_magnitud_direccion_significancia_GRANDE.png")
print(f"  • {config.OUTPUT_PATH}/7_grafico_EXTRA_GRANDE.png")
print(f"\n🎯 Tamaños de letra utilizados:")
print(f"  • Versión GRANDE: Variables=16, Valores=16, Títulos=20")
print(f"  • Versión EXTRA GRANDE: Variables=18, Valores=20, Títulos=24")


NUEVO GRÁFICO: MAGNITUD, DIRECCIÓN Y SIGNIFICANCIA (LETRAS GRANDES)
📊 Generando gráfico con letras grandes para mejor visualización...
✓ Gráfico con letras grandes guardado

📋 RESUMEN ESTADÍSTICO DEL GRÁFICO:
  - Variables mostradas: 4
  - Correlación más fuerte: 0.390
  - Correlación más débil: 0.121
  - Variables que aumentan ausentismo: 3
  - Variables que disminuyen ausentismo: 1

🔍 DETALLES POR VARIABLE:
  • Social_drinker: AUMENTA (ρ=0.121, p=0.0006)
  • Son: AUMENTA (ρ=0.132, p=0.0002)
  • Transportation_expense: AUMENTA (ρ=0.170, p=0.0000)
  • Disciplinary_failure: DISMINUYE (ρ=-0.390, p=0.0000)

📊 Generando versión EXTRA GRANDE para presentaciones...
✓ Versión EXTRA GRANDE guardada

📁 Gráficos guardados en:
  • G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3/6_grafico_magnitud_direccion_significancia_GRANDE.png
  • G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador E

In [34]:
print("\n" + "="*80)
print("GRÁFICO MEJORADO: LETRAS GRANDES Y SIN SUPERPOSICIÓN")
print("="*80)

def generar_grafico_optimizado(df_results, output_path):
    """Generar gráfico optimizado con letras grandes y sin superposición"""
    print("📊 Generando gráfico optimizado...")
    
    if df_results.empty:
        print("⚠ No hay resultados para generar el gráfico")
        return
    
    try:
        # Filtrar solo variables significativas
        df_significativas = df_results[df_results['Significativa_Spearman']].copy()
        
        if df_significativas.empty:
            print("⚠ No hay variables significativas para el gráfico")
            return
        
        # Ordenar por correlación absoluta (magnitud)
        df_plot = df_significativas.sort_values('abs_corr', ascending=True)
        
        # Crear figura MÁS ALTA para dar espacio a las etiquetas
        n_vars = len(df_plot)
        fig_height = max(10, n_vars * 0.8)  # Altura dinámica basada en número de variables
        plt.figure(figsize=(16, fig_height))
        
        # Configurar colores
        colors = []
        for _, row in df_plot.iterrows():
            if row['Correlacion_Spearman'] > 0:
                color = (0.8, 0.2, 0.2, 0.8)  # Rojo para positivo
            else:
                color = (0.2, 0.4, 0.8, 0.8)  # Azul para negativo
            colors.append(color)
        
        # Crear barras horizontales
        bar_height = 0.6
        y_pos = np.arange(len(df_plot))
        bars = plt.barh(y_pos, df_plot['Correlacion_Spearman'], 
                       height=bar_height, color=colors, alpha=0.8, 
                       edgecolor='black', linewidth=1.5)
        
        # Añadir línea vertical en 0
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=2)
        
        # AÑADIR VALORES DE MAGNITUD CON MEJOR POSICIONAMIENTO
        for i, (value, p_valor) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['p_value_Spearman'])):
            # Determinar símbolo de significancia
            if p_valor < 0.001:
                simbolo = '***'
            elif p_valor < 0.01:
                simbolo = '**'
            elif p_valor < 0.05:
                simbolo = '*'
            else:
                simbolo = ''
            
            # DECIDIR SI PONER EL TEXTO DENTRO O FUERA DE LA BARRA
            bar_width = abs(value)
            text_inside = bar_width > 0.1  # Si la barra es ancha, poner texto dentro
            
            if text_inside:
                # Texto DENTRO de la barra (color blanco para contraste)
                if value >= 0:
                    x_pos = value - 0.05  # Un poco antes del final
                    ha = 'right'
                    color_texto = 'white'
                else:
                    x_pos = value + 0.05  # Un poco después del inicio
                    ha = 'left'
                    color_texto = 'white'
            else:
                # Texto FUERA de la barra (color negro)
                if value >= 0:
                    x_pos = value + 0.02
                    ha = 'left'
                else:
                    x_pos = value - 0.02
                    ha = 'right'
                color_texto = 'black'
            
            texto = f'{value:.3f}{simbolo}'
            plt.text(x_pos, i, texto, va='center', ha=ha, fontweight='bold', 
                    fontsize=16, color=color_texto,
                    bbox=dict(boxstyle="round,pad=0.4", facecolor='white' if not text_inside else 'none', 
                             alpha=0.9 if not text_inside else 0.0, edgecolor='gray' if not text_inside else 'none'))
        
        # CONFIGURAR ETIQUETAS DE VARIABLES EN EL EJE Y CON LETRAS MUY GRANDES
        plt.yticks(y_pos, df_plot['Variable'], fontsize=18, fontweight='bold')  # Aumentado a 18 y negrita
        
        # Configurar ejes
        plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=18, fontweight='bold')
        plt.ylabel('Variables', fontsize=18, fontweight='bold')
        plt.xticks(fontsize=14)
        
        # Título
        plt.title('IMPACTO EN AUSENTISMO LABORAL\n(Magnitud, Dirección y Significancia Estadística)', 
                 fontsize=20, fontweight='bold', pad=30)
        
        # Añadir leyenda
        legend_elements = [
            plt.Rectangle((0,0), 1, 1, facecolor='red', alpha=0.7, edgecolor='black', label='Aumenta Ausentismo'),
            plt.Rectangle((0,0), 1, 1, facecolor='blue', alpha=0.7, edgecolor='black', label='Disminuye Ausentismo')
        ]
        plt.legend(handles=legend_elements, loc='lower right', framealpha=0.9, fontsize=14)
        
        # Añadir grid
        plt.grid(axis='x', alpha=0.3, linestyle='--')
        
        # Ajustar límites y márgenes
        x_margin = max(abs(df_plot['Correlacion_Spearman'])) * 0.2
        plt.xlim(min(df_plot['Correlacion_Spearman']) - x_margin, 
                max(df_plot['Correlacion_Spearman']) + x_margin)
        
        # Ajustar márgenes para que las etiquetas no se corten
        plt.subplots_adjust(left=0.4, right=0.95, top=0.95, bottom=0.08)
        
        # Mejorar estética
        plt.tight_layout()
        
        # Guardar
        plt.savefig(f'{output_path}/6_grafico_optimizado.png', 
                   dpi=350, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print("✓ Gráfico optimizado guardado")
        
        # Mostrar resumen
        print(f"\n📋 RESUMEN DEL GRÁFICO OPTIMIZADO:")
        print(f"  - Altura de figura: {fig_height:.1f} pulgadas")
        print(f"  - Tamaño letras variables: 18 puntos")
        print(f"  - Tamaño letras valores: 16 puntos")
        print(f"  - Variables mostradas: {len(df_plot)}")
        
    except Exception as e:
        print(f"✗ Error generando gráfico optimizado: {e}")

# VERSIÓN ALTERNATIVA CON DISEÑO DIFERENTE
def generar_grafico_alternativo(df_results, output_path):
    """Versión alternativa con diseño diferente para evitar superposición"""
    print("\n📊 Generando versión alternativa...")
    
    if df_results.empty:
        return
    
    try:
        df_significativas = df_results[df_results['Significativa_Spearman']].copy()
        if df_significativas.empty:
            return
        
        df_plot = df_significativas.sort_values('abs_corr', ascending=True)
        
        # Figura más alta
        n_vars = len(df_plot)
        fig_height = max(12, n_vars * 1.0)
        plt.figure(figsize=(18, fig_height))
        
        # Usar puntos en lugar de barras para evitar superposición
        y_pos = np.arange(len(df_plot))
        colors = ['red' if x > 0 else 'blue' for x in df_plot['Correlacion_Spearman']]
        sizes = [abs(x) * 2000 + 500 for x in df_plot['Correlacion_Spearman']]  # Tamaño proporcional a magnitud
        
        # Crear scatter plot
        scatter = plt.scatter(df_plot['Correlacion_Spearman'], y_pos, 
                             c=colors, s=sizes, alpha=0.7, edgecolors='black', linewidths=2)
        
        # Línea vertical en 0
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=2)
        
        # AÑADIR VALORES Y VARIABLES CON LETRAS GRANDES
        for i, (value, variable, p_valor) in enumerate(zip(df_plot['Correlacion_Spearman'], 
                                                          df_plot['Variable'], 
                                                          df_plot['p_value_Spearman'])):
            # Símbolo de significancia
            if p_valor < 0.001:
                simbolo = '***'
            elif p_valor < 0.01:
                simbolo = '**'
            elif p_valor < 0.05:
                simbolo = '*'
            else:
                simbolo = ''
            
            # NOMBRE DE LA VARIABLE A LA IZQUIERDA (MUY GRANDE)
            plt.text(-0.5, i, variable, va='center', ha='right', fontsize=16, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='lightgray', alpha=0.7))
            
            # VALOR A LA DERECHA (GRANDE)
            plt.text(0.5, i, f'{value:.3f}{simbolo}', va='center', ha='left', fontsize=14, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9))
        
        # Configurar ejes
        plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=18, fontweight='bold')
        plt.ylabel('Variables', fontsize=18, fontweight='bold')
        plt.yticks([])  # Ocultar ticks del eje Y porque tenemos las etiquetas
        
        # Límites
        x_margin = max(abs(df_plot['Correlacion_Spearman'])) * 0.3
        plt.xlim(min(df_plot['Correlacion_Spearman']) - x_margin, 
                max(df_plot['Correlacion_Spearman']) + x_margin)
        plt.ylim(-0.5, len(df_plot) - 0.5)
        
        # Título
        plt.title('IMPACTO EN AUSENTISMO LABORAL\n(Tamaño del punto = Magnitud del efecto)', 
                 fontsize=20, fontweight='bold', pad=30)
        
        # Leyenda
        legend_elements = [
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=15, label='Aumenta Ausentismo'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=15, label='Disminuye Ausentismo')
        ]
        plt.legend(handles=legend_elements, loc='lower right', fontsize=14)
        
        # Grid
        plt.grid(axis='x', alpha=0.3, linestyle='--')
        plt.grid(axis='y', alpha=0.1, linestyle='--')
        
        # Ajustar márgenes
        plt.subplots_adjust(left=0.3, right=0.95, top=0.95, bottom=0.08)
        
        plt.tight_layout()
        plt.savefig(f'{output_path}/7_grafico_alternativo.png', dpi=350, bbox_inches='tight')
        plt.close()
        
        print("✓ Versión alternativa guardada")
        
    except Exception as e:
        print(f"✗ Error en versión alternativa: {e}")

# VERSIÓN SIMPLE Y CLARA
def generar_grafico_simple_claro(df_results, output_path):
    """Versión simple y clara con máximo enfoque en legibilidad"""
    print("\n📊 Generando versión simple y clara...")
    
    if df_results.empty:
        return
    
    try:
        df_significativas = df_results[df_results['Significativa_Spearman']].copy()
        if df_significativas.empty:
            return
        
        df_plot = df_significativas.sort_values('Correlacion_Spearman', ascending=True)
        
        # Figura simple
        n_vars = len(df_plot)
        plt.figure(figsize=(14, n_vars * 0.7 + 4))
        
        # Barras simples
        colors = ['red' if x > 0 else 'blue' for x in df_plot['Correlacion_Spearman']]
        y_pos = np.arange(len(df_plot))
        
        bars = plt.barh(y_pos, df_plot['Correlacion_Spearman'], color=colors, alpha=0.7, height=0.6)
        
        # Línea en cero
        plt.axvline(x=0, color='black', linewidth=1, alpha=0.5)
        
        # ETIQUETAS DE VARIABLES MUY GRANDES Y CLARAS
        plt.yticks(y_pos, df_plot['Variable'], fontsize=16, fontweight='bold')
        
        # Valores al lado de las barras
        for i, bar in enumerate(bars):
            width = bar.get_width()
            if width >= 0:
                plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                        f'{width:.3f}', ha='left', va='center', fontsize=14, fontweight='bold')
            else:
                plt.text(width - 0.01, bar.get_y() + bar.get_height()/2, 
                        f'{width:.3f}', ha='right', va='center', fontsize=14, fontweight='bold')
        
        # Configuración simple
        plt.xlabel('Correlación con Ausentismo', fontsize=16, fontweight='bold')
        plt.title('Variables que Afectan el Ausentismo Laboral', fontsize=18, fontweight='bold', pad=20)
        
        # Grid sutil
        plt.grid(axis='x', alpha=0.2, linestyle='--')
        
        # Ajustar márgenes para las etiquetas grandes
        plt.subplots_adjust(left=0.35, right=0.95, top=0.92, bottom=0.08)
        
        plt.tight_layout()
        plt.savefig(f'{output_path}/8_grafico_simple_claro.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✓ Versión simple y clara guardada")
        
    except Exception as e:
        print(f"✗ Error en versión simple: {e}")

# EJECUTAR LAS TRES VERSIONES
generar_grafico_optimizado(df_results_numeric, config.OUTPUT_PATH)
generar_grafico_alternativo(df_results_numeric, config.OUTPUT_PATH) 
generar_grafico_simple_claro(df_results_numeric, config.OUTPUT_PATH)

print(f"\n📁 TRES VERSIONES GUARDADAS:")
print(f"  1. {config.OUTPUT_PATH}/6_grafico_optimizado.png")
print(f"     - Diseño balanceado, texto dentro/fuera de barras")
print(f"     - Letras variables: 18pts, valores: 16pts")
print(f"  2. {config.OUTPUT_PATH}/7_grafico_alternativo.png") 
print(f"     - Diseño con puntos, separación clara de elementos")
print(f"     - Variables a la izquierda, valores a la derecha")
print(f"  3. {config.OUTPUT_PATH}/8_grafico_simple_claro.png")
print(f"     - Versión minimalista, máximo enfoque en legibilidad")
print(f"     - Diseño limpio y directo")

print(f"\n🎯 RECOMENDACIÓN:")
print(f"  • Usar '6_grafico_optimizado.png' para informes detallados")
print(f"  • Usar '8_grafico_simple_claro.png' para presentaciones ejecutivas")
print(f"  • Usar '7_grafico_alternativo.png' si prefieres diseño visual con puntos")


GRÁFICO MEJORADO: LETRAS GRANDES Y SIN SUPERPOSICIÓN
📊 Generando gráfico optimizado...
✓ Gráfico optimizado guardado

📋 RESUMEN DEL GRÁFICO OPTIMIZADO:
  - Altura de figura: 10.0 pulgadas
  - Tamaño letras variables: 18 puntos
  - Tamaño letras valores: 16 puntos
  - Variables mostradas: 4

📊 Generando versión alternativa...
✓ Versión alternativa guardada

📊 Generando versión simple y clara...
✓ Versión simple y clara guardada

📁 TRES VERSIONES GUARDADAS:
  1. G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3/6_grafico_optimizado.png
     - Diseño balanceado, texto dentro/fuera de barras
     - Letras variables: 18pts, valores: 16pts
  2. G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3/7_grafico_alternativo.png
     - Diseño con puntos, separación clara de elementos
     - Variables a l

In [35]:
print("\n" + "="*80)
print("INFORME EJECUTIVO COMPLETO - VARIABLES PERSONALES Y LABORALES")
print("="*80)

class InformeEjecutivo:
    """Clase para generar informe ejecutivo completo clasificado por tipo de variable"""
    
    @staticmethod
    def generar_informe_completo(df_results_numeric, df_results_categorical, df_results_binarias_corregido, 
                                variables_personales, variables_laborales, variables_temporales,
                                resultados_razones):
        """Generar informe ejecutivo completo clasificado por tipo de variable"""
        
        print("\n📋 INFORME EJECUTIVO COMPLETO")
        print("="*60)
        
        # 1. VARIABLES PERSONALES SIGNIFICATIVAS
        print("\n🎯 VARIABLES PERSONALES SIGNIFICATIVAS:")
        print("-" * 40)
        vars_personales_significativas = []
        for var in variables_personales:
            if var in df_results_numeric['Variable'].values:
                resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
                if resultado['Significativa_Spearman']:
                    # CORREGIDO: Extraer correctamente la fuerza
                    fuerza_texto = resultado['Fuerza_Spearman']
                    if not isinstance(fuerza_texto, str):
                        # Si es una serie, extraer el valor
                        fuerza_texto = str(fuerza_texto).split()[-1] if len(str(fuerza_texto).split()) > 1 else "Débil"
                    
                    direccion = "AUMENTA" if resultado['Correlacion_Spearman'] > 0 else "DISMINUYE"
                    
                    vars_personales_significativas.append({
                        'Variable': var,
                        'Correlacion': resultado['Correlacion_Spearman'],
                        'Direccion': direccion,
                        'Fuerza': fuerza_texto,
                        'p_valor': resultado['p_value_Spearman'],
                        'Cohens_d': resultado['Cohens_d'],
                        'Tamaño_Efecto_Cohens': resultado['Tamaño_Efecto']
                    })
                    print(f"✓ {var}: {direccion} ausentismo")
                    print(f"  - ρ = {resultado['Correlacion_Spearman']:.3f} ({fuerza_texto}), p = {resultado['p_value_Spearman']:.6f}")
                    print(f"  - Cohen's d = {resultado['Cohens_d']:.3f} ({resultado['Tamaño_Efecto']})")
        
        if not vars_personales_significativas:
            print("  No hay variables personales significativas")
        
        # 2. VARIABLES LABORALES SIGNIFICATIVAS
        print("\n💼 VARIABLES LABORALES SIGNIFICATIVAS:")
        print("-" * 40)
        vars_laborales_significativas = []
        for var in variables_laborales:
            if var in df_results_numeric['Variable'].values:
                resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
                if resultado['Significativa_Spearman']:
                    # CORREGIDO: Extraer correctamente la fuerza
                    fuerza_texto = resultado['Fuerza_Spearman']
                    if not isinstance(fuerza_texto, str):
                        # Si es una serie, extraer el valor
                        fuerza_texto = str(fuerza_texto).split()[-1] if len(str(fuerza_texto).split()) > 1 else "Débil"
                    
                    direccion = "AUMENTA" if resultado['Correlacion_Spearman'] > 0 else "DISMINUYE"
                    
                    vars_laborales_significativas.append({
                        'Variable': var,
                        'Correlacion': resultado['Correlacion_Spearman'],
                        'Direccion': direccion,
                        'Fuerza': fuerza_texto,
                        'p_valor': resultado['p_value_Spearman'],
                        'Cohens_d': resultado['Cohens_d'],
                        'Tamaño_Efecto_Cohens': resultado['Tamaño_Efecto']
                    })
                    print(f"✓ {var}: {direccion} ausentismo")
                    print(f"  - ρ = {resultado['Correlacion_Spearman']:.3f} ({fuerza_texto}), p = {resultado['p_value_Spearman']:.6f}")
                    print(f"  - Cohen's d = {resultado['Cohens_d']:.3f} ({resultado['Tamaño_Efecto']})")
        
        if not vars_laborales_significativas:
            print("  No hay variables laborales significativas")
        
        # 3. ANÁLISIS CORREGIDO DE RAZONES DE AUSENTISMO
        print("\n🏥 ANÁLISIS CORREGIDO - RAZONES DE AUSENTISMO:")
        print("-" * 50)
        
        print("📊 RESULTADOS POR TIPO DE RAZÓN (CORREGIDO):")
        stats_razones = resultados_razones['stats_por_razon']
        for razon, fila in stats_razones.iterrows():
            print(f"  • {razon}:")
            print(f"    - n: {fila['count']}, media: {fila['mean']:.1f}h, mediana: {fila['median']:.1f}h")
        
        print(f"\n🔬 SIGNIFICANCIA ESTADÍSTICA:")
        print(f"  - Kruskal-Wallis: p = {resultados_razones['kw_pvalue']:.6f}")
        print(f"  - Chi-cuadrado: p = {resultados_razones['chi2_pvalue']:.6f}")
        
        print(f"\n💡 INTERPRETACIÓN CORRECTA:")
        print(f"  - El TIPO de razón afecta significativamente la duración del ausentismo")
        print(f"  - NO existe relación ordinal entre códigos numéricos y horas")
        print(f"  - Diferencias reales entre tipos (ej: enfermedades vs consultas)")
        
        # 4. VARIABLES BINARIAS SIGNIFICATIVAS
        print("\n🔘 VARIABLES BINARIAS SIGNIFICATIVAS:")
        print("-" * 40)
        if df_results_binarias_corregido is not None and not df_results_binarias_corregido.empty:
            for _, row in df_results_binarias_corregido[df_results_binarias_corregido['Significativa']].iterrows():
                # Clasificar si es personal o laboral
                tipo = "PERSONAL" if row['Variable'] in ['Social_drinker', 'Social_smoker'] else "LABORAL"
                print(f"✓ {row['Variable']} ({tipo}): {row['Interpretacion_Direccion']} ausentismo")
                print(f"   - Test: {row['Test_Utilizado']}, p = {row['p_valor']:.6f}")
                print(f"   - Diferencia: {row['Media_Grupo1']:.2f} vs {row['Media_Grupo0']:.2f} horas")
                print(f"   - Cohen's d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
                if not np.isnan(row['PointBiserial_Corr']):
                    print(f"   - Correlación Punto-Biserial: {row['PointBiserial_Corr']:.3f} ({row['Fuerza_PointBiserial']})")
                if not np.isnan(row['Odds_Ratio']):
                    print(f"   - Odds Ratio: {row['Odds_Ratio']:.3f}")
        
        # 5. RESUMEN PARA REGRESIONES MULTIVARIABLES
        print("\n📈 VARIABLES CANDIDATAS PARA REGRESIONES MULTIVARIABLES:")
        print("-" * 50)
        
        # Lista de variables significativas (excluyendo Reason_absence_numeric)
        todas_significativas = vars_personales_significativas + vars_laborales_significativas
        
        if todas_significativas:
            # Ordenar por fuerza de correlación absoluta
            todas_significativas.sort(key=lambda x: abs(x['Correlacion']), reverse=True)
            
            print("Variables recomendadas para modelo multivariable (ordenadas por importancia):")
            for i, var in enumerate(todas_significativas[:10], 1):
                print(f"{i}. {var['Variable']}:")
                print(f"   - ρ = {var['Correlacion']:.3f} ({var['Fuerza']}, {var['Direccion']})")
                print(f"   - Cohen's d = {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})")
                print(f"   - p = {var['p_valor']:.6f}")
        
        # Añadir la variable de razones agrupadas como especial
        print(f"\n🎯 VARIABLE ESPECIAL - RAZONES AGRUPADAS:")
        print(f"  - Kruskal-Wallis: p = {resultados_razones['kw_pvalue']:.6f}")
        print(f"  - Tipo: Variable categórica (agrupamiento lógico)")
        print(f"  - Interpretación: El tipo de razón explica diferencias en duración de ausentismo")
        
        print(f"\n📊 RESUMEN ESTADÍSTICO:")
        print(f"   - Total variables significativas: {len(todas_significativas)}")
        print(f"   - Variables personales: {len(vars_personales_significativas)}")
        print(f"   - Variables laborales: {len(vars_laborales_significativas)}")
        print(f"   - Variable razones agrupadas: 1 (análisis categórico corregido)")
        
        # 6. GUARDAR INFORME DETALLADO
        InformeEjecutivo.guardar_informe_detallado(vars_personales_significativas, vars_laborales_significativas, 
                                                  df_results_binarias_corregido, resultados_razones, config.OUTPUT_PATH)
    
    @staticmethod
    def guardar_informe_detallado(vars_personales, vars_laborales, df_binarias, resultados_razones, output_path):
        """Guardar informe detallado en archivo"""
        
        with open(f'{output_path}/INFORME_EJECUTIVO_DETALLADO.txt', 'w', encoding='utf-8') as f:
            f.write("INFORME EJECUTIVO DETALLADO - ANÁLISIS DE AUSENTISMO LABORAL\n")
            f.write("="*70 + "\n\n")
            
            f.write("RESUMEN EJECUTIVO:\n")
            f.write(f"- Variables personales significativas: {len(vars_personales)}\n")
            f.write(f"- Variables laborales significativas: {len(vars_laborales)}\n")
            if df_binarias is not None:
                binarias_significativas = len(df_binarias[df_binarias['Significativa']]) if not df_binarias.empty else 0
                f.write(f"- Variables binarias significativas: {binarias_significativas}\n")
            f.write(f"- Variable razones agrupadas: 1 (análisis categórico corregido)\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("VARIABLES PERSONALES SIGNIFICATIVAS:\n")
            f.write("="*70 + "\n")
            for var in vars_personales:
                f.write(f"✓ {var['Variable']}: {var['Direccion']} el ausentismo\n")
                f.write(f"  - Correlación (Spearman): {var['Correlacion']:.3f} ({var['Fuerza']})\n")
                f.write(f"  - p-valor: {var['p_valor']:.6f}\n")
                f.write(f"  - Cohen's d: {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})\n")
                f.write(f"  - Interpretación: {'Mayor valor de esta variable se asocia con ' + var['Direccion'].lower() + ' ausentismo'}\n\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("VARIABLES LABORALES SIGNIFICATIVAS:\n")
            f.write("="*70 + "\n")
            for var in vars_laborales:
                f.write(f"✓ {var['Variable']}: {var['Direccion']} el ausentismo\n")
                f.write(f"  - Correlación (Spearman): {var['Correlacion']:.3f} ({var['Fuerza']})\n")
                f.write(f"  - p-valor: {var['p_valor']:.6f}\n")
                f.write(f"  - Cohen's d: {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})\n")
                f.write(f"  - Interpretación: {'Mayor valor de esta variable se asocia con ' + var['Direccion'].lower() + ' ausentismo'}\n\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("ANÁLISIS CORREGIDO - RAZONES DE AUSENTISMO:\n")
            f.write("="*70 + "\n")
            f.write("RESULTADOS POR TIPO DE RAZÓN (CORREGIDO):\n\n")
            stats_razones = resultados_razones['stats_por_razon']
            for razon, fila in stats_razones.iterrows():
                f.write(f"• {razon}:\n")
                f.write(f"  - n: {fila['count']}, media: {fila['mean']:.1f}h, mediana: {fila['median']:.1f}h\n")
            
            f.write(f"\nSIGNIFICANCIA ESTADÍSTICA:\n")
            f.write(f"- Kruskal-Wallis: p = {resultados_razones['kw_pvalue']:.6f}\n")
            f.write(f"- Chi-cuadrado: p = {resultados_razones['chi2_pvalue']:.6f}\n")
            
            f.write(f"\nINTERPRETACIÓN CORRECTA:\n")
            f.write(f"- El TIPO de razón afecta significativamente la duración del ausentismo\n")
            f.write(f"- NO existe relación ordinal entre códigos numéricos y horas\n")
            f.write(f"- Diferencias reales entre tipos (ej: enfermedades vs consultas)\n")
            
            if df_binarias is not None and not df_binarias.empty:
                f.write("\n" + "="*70 + "\n")
                f.write("VARIABLES BINARIAS SIGNIFICATIVAS:\n")
                f.write("="*70 + "\n")
                for _, row in df_binarias[df_binarias['Significativa']].iterrows():
                    f.write(f"✓ {row['Variable']}: {row['Interpretacion_Direccion']} el ausentismo\n")
                    f.write(f"  - Test utilizado: {row['Test_Utilizado']}\n")
                    f.write(f"  - p-valor: {row['p_valor']:.6f}\n")
                    f.write(f"  - Diferencia de medias: {row['Media_Grupo1']:.2f} vs {row['Media_Grupo0']:.2f} horas\n")
                    f.write(f"  - Tamaño del efecto (Cohen's d): {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                    if not np.isnan(row['PointBiserial_Corr']):
                        f.write(f"  - Correlación Punto-Biserial: {row['PointBiserial_Corr']:.3f} ({row['Fuerza_PointBiserial']})\n")
                    if not np.isnan(row['Odds_Ratio']):
                        f.write(f"  - Odds Ratio: {row['Odds_Ratio']:.3f}\n")
                    if not np.isnan(row['Risk_Ratio']):
                        f.write(f"  - Risk Ratio: {row['Risk_Ratio']:.3f}\n\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("RECOMENDACIONES PARA ANÁLISIS MULTIVARIABLE:\n")
            f.write("="*70 + "\n")
            f.write("Variables recomendadas para regresiones lineales multivariables:\n")
            todas_vars = vars_personales + vars_laborales
            todas_vars.sort(key=lambda x: abs(x['Correlacion']), reverse=True)
            
            for i, var in enumerate(todas_vars[:10], 1):
                f.write(f"{i}. {var['Variable']}\n")
                f.write(f"   - Correlación: ρ = {var['Correlacion']:.3f} ({var['Direccion']})\n")
                f.write(f"   - Tamaño efecto: Cohen's d = {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})\n")
                f.write(f"   - Significancia: p = {var['p_valor']:.6f}\n")
            
            f.write(f"\nVARIABLE ESPECIAL - RAZONES AGRUPADAS:\n")
            f.write(f"- Kruskal-Wallis: p = {resultados_razones['kw_pvalue']:.6f}\n")
            f.write(f"- Tipo: Variable categórica (agrupamiento lógico)\n")
            f.write(f"- Interpretación: El tipo de razón explica diferencias en duración de ausentismo\n")
            
            f.write("\nConsideraciones para el modelo multivariable:\n")
            f.write("- Evaluar multicolinealidad entre variables predictoras\n")
            f.write("- Considerar interacciones entre variables personales y laborales\n")
            f.write("- Validar supuestos de normalidad y homocedasticidad\n")
            f.write("- Considerar transformaciones si es necesario\n")
            f.write("- Priorizar variables con mayor Cohen's d para mayor poder explicativo\n")
            f.write("- Usar la variable 'Razon_agrupada' en lugar de 'Reason_absence_numeric'\n")

# Ejecutar informe ejecutivo
InformeEjecutivo.generar_informe_completo(
    df_results_numeric,
    df_results_categorical, 
    df_results_binarias_corregido,
    VARIABLES_PERSONALES,
    VARIABLES_LABORALES, 
    VARIABLES_TEMPORALES,
    resultados_razones  # Nuevo parámetro
)

print(f"\n📄 Informe ejecutivo detallado guardado en: {config.OUTPUT_PATH}/INFORME_EJECUTIVO_DETALLADO.txt")


INFORME EJECUTIVO COMPLETO - VARIABLES PERSONALES Y LABORALES

📋 INFORME EJECUTIVO COMPLETO

🎯 VARIABLES PERSONALES SIGNIFICATIVAS:
----------------------------------------
✓ Son: AUMENTA ausentismo
  - ρ = 0.132 (object), p = 0.000169
  - Cohen's d = 0.226 (Pequeño)
✓ Social_drinker: AUMENTA ausentismo
  - ρ = 0.121 (object), p = 0.000602
  - Cohen's d = 0.374 (Pequeño)

💼 VARIABLES LABORALES SIGNIFICATIVAS:
----------------------------------------
✓ Transportation_expense: AUMENTA ausentismo
  - ρ = 0.170 (object), p = 0.000001
  - Cohen's d = 0.032 (Muy pequeño)
✓ Disciplinary_failure: DISMINUYE ausentismo
  - ρ = -0.390 (object), p = 0.000000
  - Cohen's d = -0.259 (Pequeño)

🏥 ANÁLISIS CORREGIDO - RAZONES DE AUSENTISMO:
--------------------------------------------------
📊 RESULTADOS POR TIPO DE RAZÓN (CORREGIDO):
  • Ausencias no justificadas:
    - n: 35.0, media: 7.3h, mediana: 8.0h
  • Condiciones diversas:
    - n: 80.0, media: 14.7h, mediana: 8.0h
  • Consulta médica:
    - 

9. Exportación de Resultados

In [36]:
print("\n" + "="*80)
print("FASE 9: EXPORTACIÓN DE RESULTADOS (ACTUALIZADA)")
print("="*80)

class ExportadorResultados:
    """Clase para exportar resultados de manera organizada - VERSIÓN ACTUALIZADA"""
    
    def __init__(self, output_path):
        self.output_path = output_path
    
    def exportar_resultados_completos(self, df_numeric, df_categorical, df_normality, df_clean, 
                                    df_binarias_corregido, df_or_rr_corregidos=None, df_transformaciones=None,
                                    correlaciones_problematicas=None):
        """Exportar todos los resultados incluyendo las correcciones"""
        print("💾 Exportando resultados completos (INCLUYENDO CORRECCIONES)...")
        
        try:
            # Crear directorio si no existe
            os.makedirs(self.output_path, exist_ok=True)
            
            # Exportar DataFrames individuales ORIGINALES
            print("  📁 Exportando resultados originales...")
            df_numeric.to_csv(f'{self.output_path}/resultados_numericos.csv', index=False, encoding='utf-8-sig')
            df_categorical.to_csv(f'{self.output_path}/resultados_categoricos.csv', index=False, encoding='utf-8-sig')
            df_normality.to_csv(f'{self.output_path}/analisis_normalidad.csv', index=False)
            
            if df_binarias_corregido is not None:
                df_binarias_corregido.to_csv(f'{self.output_path}/analisis_binarias_corregido.csv', index=False, encoding='utf-8-sig')
            
            df_clean.to_parquet(f'{self.output_path}/dataset_analisis.parquet', index=False)
            
            # Exportar DataFrames NUEVOS de correcciones
            print("  📁 Exportando correcciones y análisis adicionales...")
            if df_or_rr_corregidos is not None and not df_or_rr_corregidos.empty:
                df_or_rr_corregidos.to_csv(f'{self.output_path}/correcciones_or_rr.csv', index=False, encoding='utf-8-sig')
                print("    ✓ correcciones_or_rr.csv")
            
            if df_transformaciones is not None and not df_transformaciones.empty:
                df_transformaciones.to_csv(f'{self.output_path}/analisis_transformaciones.csv', index=False, encoding='utf-8-sig')
                print("    ✓ analisis_transformaciones.csv")
            
            # Exportar análisis de multicolinealidad si está disponible
            if correlaciones_problematicas is not None:
                with open(f'{self.output_path}/multicolinealidad_alertas.txt', 'w', encoding='utf-8') as f:
                    f.write("ALERTAS DE MULTICOLINEALIDAD\n")
                    f.write("="*40 + "\n\n")
                    if correlaciones_problematicas:
                        f.write("CORRELACIONES PROBLEMÁTICAS DETECTADAS:\n")
                        for corr in correlaciones_problematicas:
                            f.write(f"- {corr['Variable1']} ↔ {corr['Variable2']}: {corr['Correlación']:.3f}\n")
                    else:
                        f.write("No se detectaron correlaciones problemáticas (> 0.7)\n")
                print("    ✓ multicolinealidad_alertas.txt")
            
            # Crear reporte consolidado ACTUALIZADO
            self._crear_reporte_consolidado(df_numeric, df_categorical, df_binarias_corregido, 
                                          df_or_rr_corregidos, df_transformaciones, correlaciones_problematicas)
            
            # Crear archivo de metadatos del análisis
            self._crear_metadatos_analisis()
            
            print("✓ Todos los archivos exportados correctamente")
            
        except Exception as e:
            print(f"✗ Error en exportación: {e}")
    
    def _crear_reporte_consolidado(self, df_numeric, df_categorical, df_binarias_corregido, 
                                 df_or_rr_corregidos, df_transformaciones, correlaciones_problematicas):
        """Crear reporte ejecutivo consolidado ACTUALIZADO"""
        print("📋 Generando reporte ejecutivo actualizado...")
        
        # Variables significativas
        vars_significativas = df_numeric[df_numeric['Significativa_Spearman']]
        vars_positivas = vars_significativas[vars_significativas['Correlacion_Spearman'] > 0]
        vars_negativas = vars_significativas[vars_significativas['Correlacion_Spearman'] < 0]
        
        with open(f'{self.output_path}/REPORTE_EJECUTIVO_ACTUALIZADO.txt', 'w', encoding='utf-8') as f:
            f.write("REPORTE EJECUTIVO ACTUALIZADO - ANÁLISIS DE AUSENTISMO LABORAL\n")
            f.write("="*70 + "\n\n")
            
            f.write("RESUMEN EJECUTIVO CON CORRECCIONES APLICADAS:\n")
            f.write(f"- Variables personales significativas: {len(vars_positivas[vars_positivas['Variable'].isin(VARIABLES_PERSONALES)])}\n")
            f.write(f"- Variables laborales significativas: {len(vars_negativas[vars_negativas['Variable'].isin(VARIABLES_LABORALES)])}\n")
            if df_binarias_corregido is not None:
                binarias_significativas = len(df_binarias_corregido[df_binarias_corregido['Significativa']]) if not df_binarias_corregido.empty else 0
                f.write(f"- Variables binarias significativas: {binarias_significativas}\n")
            
            # Añadir información de correcciones
            if df_or_rr_corregidos is not None and not df_or_rr_corregidos.empty:
                f.write(f"- Correcciones OR/RR aplicadas: {len(df_or_rr_corregidos)} variables\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("VARIABLES NUMÉRICAS SIGNIFICATIVAS QUE AUMENTAN AUSENTISMO:\n")
            for _, row in vars_positivas.iterrows():
                f.write(f"✓ {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})\n")
                f.write(f"  - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                f.write(f"  - p-valor: {row['p_value_Spearman']:.6f}\n")
            
            f.write("\nVARIABLES NUMÉRICAS SIGNIFICATIVAS QUE DISMINUYEN AUSENTISMO:\n")
            for _, row in vars_negativas.iterrows():
                f.write(f"✓ {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})\n")
                f.write(f"  - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                f.write(f"  - p-valor: {row['p_value_Spearman']:.6f}\n")
            
            # Añadir interpretación de variables binarias CORREGIDAS
            if df_binarias_corregido is not None and not df_binarias_corregido.empty:
                f.write("\nVARIABLES BINARIAS SIGNIFICATIVAS (ANÁLISIS CORREGIDO):\n")
                for _, row in df_binarias_corregido[df_binarias_corregido['Significativa']].iterrows():
                    efecto = row['Interpretacion_Direccion']
                    f.write(f"✓ {row['Variable']}: {efecto} el ausentismo (p={row['p_valor']:.4f}, Test: {row['Test_Utilizado']})\n")
                    f.write(f"  - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                    if not np.isnan(row['Odds_Ratio']):
                        f.write(f"  - Odds Ratio: {row['Odds_Ratio']:.2f}\n")
                    if not np.isnan(row['Risk_Ratio']):
                        f.write(f"  - Risk Ratio: {row['Risk_Ratio']:.2f}\n")
            
            # Añadir información de correcciones OR/RR
            if df_or_rr_corregidos is not None and not df_or_rr_corregidos.empty:
                f.write("\n" + "="*70 + "\n")
                f.write("CORRECCIONES APLICADAS - OR/RR:\n")
                f.write("="*70 + "\n")
                for _, row in df_or_rr_corregidos.iterrows():
                    f.write(f"✓ {row['Variable']}:\n")
                    f.write(f"  - Risk Ratio corregido: {row['Risk_Ratio_Corregido']:.3f}\n")
                    f.write(f"  - Odds Ratio corregido: {row['Odds_Ratio_Corregido']:.3f}\n")
                    f.write(f"  - Diferencia de riesgo: {row['Diferencia_Riesgo']:.3f}\n")
            
            # Añadir recomendaciones de transformaciones
            if df_transformaciones is not None and not df_transformaciones.empty:
                f.write("\n" + "="*70 + "\n")
                f.write("RECOMENDACIONES DE TRANSFORMACIONES:\n")
                f.write("="*70 + "\n")
                # Encontrar mejor transformación para cada variable
                variables_unicas = df_transformaciones['Variable'].unique()
                for var in variables_unicas:
                    data_var = df_transformaciones[df_transformaciones['Variable'] == var]
                    mejor_transform = data_var.loc[data_var['Asimetría'].abs().idxmin()]
                    f.write(f"✓ {var}: {mejor_transform['Transformación']} (asimetría: {mejor_transform['Asimetría']:.3f})\n")
            
            # Añadir alertas de multicolinealidad
            if correlaciones_problematicas is not None:
                f.write("\n" + "="*70 + "\n")
                f.write("ALERTAS DE MULTICOLINEALIDAD:\n")
                f.write("="*70 + "\n")
                if correlaciones_problematicas:
                    f.write("CORRELACIONES PROBLEMÁTICAS DETECTADAS:\n")
                    for corr in correlaciones_problematicas:
                        f.write(f"⚠ {corr['Variable1']} ↔ {corr['Variable2']}: {corr['Correlación']:.3f}\n")
                    f.write("\nRECOMENDACIÓN: Considerar eliminar una variable de cada par correlacionado\n")
                else:
                    f.write("✅ No se detectaron problemas de multicolinealidad\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("RECOMENDACIONES PARA ANÁLISIS MULTIVARIABLE:\n")
            f.write("="*70 + "\n")
            
            # Variables recomendadas ordenadas por Cohen's d
            todas_significativas = []
            for _, row in vars_significativas.iterrows():
                todas_significativas.append({
                    'Variable': row['Variable'],
                    'Correlacion': row['Correlacion_Spearman'],
                    'Cohens_d': row['Cohens_d'],
                    'Tamaño_Efecto': row['Tamaño_Efecto']
                })
            
            # Ordenar por valor absoluto de Cohen's d
            todas_significativas.sort(key=lambda x: abs(x['Cohens_d']), reverse=True)
            
            f.write("Variables recomendadas para regresiones lineales multivariables:\n")
            for i, var in enumerate(todas_significativas[:10], 1):
                direccion = "AUMENTA" if var['Correlacion'] > 0 else "DISMINUYE"
                f.write(f"{i}. {var['Variable']}\n")
                f.write(f"   - Correlación: ρ = {var['Correlacion']:.3f} ({direccion})\n")
                f.write(f"   - Tamaño efecto: Cohen's d = {var['Cohens_d']:.3f} ({var['Tamaño_Efecto']})\n")
            
            f.write("\nConsideraciones para el modelo multivariable:\n")
            f.write("- Usar OR/RR corregidos para interpretación\n")
            f.write("- Aplicar transformaciones recomendadas\n")
            f.write("- Evaluar multicolinealidad entre variables predictoras\n")
            f.write("- Considerar interacciones entre variables personales y laborales\n")
            f.write("- Priorizar variables con mayor Cohen's d para mayor poder explicativo\n")
    
    def _crear_metadatos_analisis(self):
        """Crear archivo de metadatos del análisis"""
        with open(f'{self.output_path}/METADATOS_ANALISIS.txt', 'w', encoding='utf-8') as f:
            f.write("METADATOS DEL ANÁLISIS - AUSENTISMO LABORAL\n")
            f.write("="*50 + "\n\n")
            f.write(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total de observaciones: {len(df_clean):,}\n")
            f.write(f"Variables personales analizadas: {len(VARIABLES_PERSONALES)}\n")
            f.write(f"Variables laborales analizadas: {len(VARIABLES_LABORALES)}\n")
            f.write(f"Variables binarias analizadas: {len(VARIABLES_BINARIAS)}\n")
            f.write(f"Umbral de significancia: p < 0.05\n")
            f.write(f"Métodos estadísticos: Spearman, Mann-Whitney, Kruskal-Wallis\n")
            f.write(f"Correcciones aplicadas: OR/RR, transformaciones, multicolinealidad\n")

# Llamada ACTUALIZADA con todos los parámetros nuevos
print("📤 Exportando resultados con correcciones aplicadas...")
exportador = ExportadorResultados(config.OUTPUT_PATH)

# Asegurarse de que las variables nuevas existen (si no, pasar None)
try:
    df_or_rr_corregidos_var = df_or_rr_corregidos if 'df_or_rr_corregidos' in locals() else None
except:
    df_or_rr_corregidos_var = None

try:
    df_transformaciones_var = df_transformaciones if 'df_transformaciones' in locals() else None
except:
    df_transformaciones_var = None

try:
    correlaciones_problematicas_var = correlaciones_problematicas if 'correlaciones_problematicas' in locals() else None
except:
    correlaciones_problematicas_var = None

exportador.exportar_resultados_completos(
    df_results_numeric, 
    df_results_categorical, 
    df_normality, 
    df_clean,
    df_results_binarias_corregido,
    df_or_rr_corregidos_var,        # NUEVO: resultados OR/RR corregidos
    df_transformaciones_var,         # NUEVO: análisis de transformaciones
    correlaciones_problematicas_var  # NUEVO: análisis de multicolinealidad
)

print(f"📁 Resultados exportados en: {config.OUTPUT_PATH}")
print("✓ resultados_numericos.csv")
print("✓ resultados_categoricos.csv") 
print("✓ analisis_binarias_corregido.csv")
print("✓ correcciones_or_rr.csv (NUEVO)")
print("✓ analisis_transformaciones.csv (NUEVO)")
print("✓ multicolinealidad_alertas.txt (NUEVO)")
print("✓ REPORTE_EJECUTIVO_ACTUALIZADO.txt (NUEVO)")
print("✓ METADATOS_ANALISIS.txt (NUEVO)")


FASE 9: EXPORTACIÓN DE RESULTADOS (ACTUALIZADA)
📤 Exportando resultados con correcciones aplicadas...
💾 Exportando resultados completos (INCLUYENDO CORRECCIONES)...
  📁 Exportando resultados originales...
  📁 Exportando correcciones y análisis adicionales...
    ✓ correcciones_or_rr.csv
    ✓ analisis_transformaciones.csv
    ✓ multicolinealidad_alertas.txt
📋 Generando reporte ejecutivo actualizado...
✓ Todos los archivos exportados correctamente
📁 Resultados exportados en: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3
✓ resultados_numericos.csv
✓ resultados_categoricos.csv
✓ analisis_binarias_corregido.csv
✓ correcciones_or_rr.csv (NUEVO)
✓ analisis_transformaciones.csv (NUEVO)
✓ multicolinealidad_alertas.txt (NUEVO)
✓ REPORTE_EJECUTIVO_ACTUALIZADO.txt (NUEVO)
✓ METADATOS_ANALISIS.txt (NUEVO)


In [37]:
print("\n" + "="*80)
print("ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO")
print("="*80)

# Métricas finales
vars_analizadas = len(VARIABLES_PERSONALES) + len(VARIABLES_LABORALES)  # Excluimos temporales
vars_significativas_numeric = len(df_results_numeric[df_results_numeric['Significativa_Spearman']])
vars_significativas_categorical = df_results_categorical['Significativa_Principal'].sum()
vars_significativas_binarias = len(df_results_binarias_corregido[df_results_binarias_corregido['Significativa']]) if 'df_results_binarias_corregido' in locals() else 0

# Contar variables personales y laborales significativas
vars_personales_significativas = [var for var in VARIABLES_PERSONALES 
                                 if var in df_results_numeric['Variable'].values 
                                 and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

vars_laborales_significativas = [var for var in VARIABLES_LABORALES 
                                if var in df_results_numeric['Variable'].values 
                                and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

print(f"📊 MÉTRICAS DEL ANÁLISIS:")
print(f"• Variables analizadas (personales + laborales): {vars_analizadas}")
print(f"• Variables personales significativas: {len(vars_personales_significativas)}")
print(f"• Variables laborales significativas: {len(vars_laborales_significativas)}")
print(f"• Variables binarias significativas: {vars_significativas_binarias}")
print(f"• Variables temporales (excluidas): {len(VARIABLES_TEMPORALES)}")
print(f"• Ruta de resultados: {config.OUTPUT_PATH}")

# Distribución de categorías de ausentismo
distribucion_ausentismo = df_clean[TARGET_CATEGORICO].value_counts()
print(f"\n📈 DISTRIBUCIÓN DE CATEGORÍAS DE AUSENTISMO:")
for categoria, count in distribucion_ausentismo.items():
    porcentaje = (count / len(df_clean)) * 100
    print(f"  • {categoria}: {count} empleados ({porcentaje:.1f}%)")

# Variables más importantes (solo significativas personales y laborales)
if not df_results_numeric.empty:
    variables_significativas = df_results_numeric[
        df_results_numeric['Significativa_Spearman'] & 
        (df_results_numeric['Variable'].isin(VARIABLES_PERSONALES + VARIABLES_LABORALES))
    ]
    if not variables_significativas.empty:
        top_variables = variables_significativas.nlargest(5, 'abs_corr')
        print(f"\n🏆 TOP 5 VARIABLES MÁS IMPORTANTES (PERSONALES Y LABORALES):")
        for i, (_, row) in enumerate(top_variables.iterrows(), 1):
            direccion = "Aumenta" if row['Correlacion_Spearman'] > 0 else "Disminuye"
            # Clasificar tipo
            tipo = "PERSONAL" if row['Variable'] in VARIABLES_PERSONALES else "LABORAL"
            print(f"{i}. {row['Variable']} ({tipo}):")
            print(f"   - ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})")
            print(f"   - Cohen's d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
            print(f"   - {direccion} ausentismo")

# Interpretación contextual mejorada
def interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido):
    print(f"\n🔍 HALLAZGOS CLAVE POR TIPO DE VARIABLE:")
    
    print("\n  🎯 VARIABLES PERSONALES SIGNIFICATIVAS:")
    personales_significativas = df_results_numeric[
        df_results_numeric['Significativa_Spearman'] & 
        (df_results_numeric['Variable'].isin(VARIABLES_PERSONALES))
    ]
    
    if not personales_significativas.empty:
        for _, row in personales_significativas.iterrows():
            efecto = "aumenta" if row['Correlacion_Spearman'] > 0 else "disminuye"
            print(f"    • {row['Variable']}: {efecto} el ausentismo")
            print(f"      - Correlación: ρ = {row['Correlacion_Spearman']:.3f}")
            print(f"      - Tamaño efecto: d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
    else:
        print("    No hay variables personales significativas")
    
    print("\n  💼 VARIABLES LABORALES SIGNIFICATIVAS:")
    laborales_significativas = df_results_numeric[
        df_results_numeric['Significativa_Spearman'] & 
        (df_results_numeric['Variable'].isin(VARIABLES_LABORALES))
    ]
    
    if not laborales_significativas.empty:
        for _, row in laborales_significativas.iterrows():
            efecto = "aumenta" if row['Correlacion_Spearman'] > 0 else "disminuye"
            print(f"    • {row['Variable']}: {efecto} el ausentismo")
            print(f"      - Correlación: ρ = {row['Correlacion_Spearman']:.3f}")
            print(f"      - Tamaño efecto: d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
    else:
        print("    No hay variables laborales significativas")
    
    # Hallazgos binarios CORREGIDOS
    if df_results_binarias_corregido is not None:
        print("\n  🔘 VARIABLES BINARIAS SIGNIFICATIVAS:")
        binarias_significativas = df_results_binarias_corregido[df_results_binarias_corregido['Significativa']]
        if not binarias_significativas.empty:
            for _, row in binarias_significativas.iterrows():
                tipo = "PERSONAL" if row['Variable'] in ['Social_drinker', 'Social_smoker'] else "LABORAL"
                print(f"    • {row['Variable']} ({tipo}): {row['Interpretacion_Direccion']} ausentismo")
                print(f"      - Test: {row['Test_Utilizado']}, p = {row['p_valor']:.4f}")
                print(f"      - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
        else:
            print("    No hay variables binarias significativas")

interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido)

print(f"\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE!")
print(f"📁 Resultados guardados en: {config.OUTPUT_PATH}")
print(f"📄 Informe ejecutivo: {config.OUTPUT_PATH}/INFORME_EJECUTIVO_DETALLADO.txt")
print(f"🔮 Próximo paso: Usar variables significativas para regresiones lineales multivariables")
print(f"💡 Considerar: Variables con Cohen's d > 0.5 tienen efecto mediano/grande")


ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO
📊 MÉTRICAS DEL ANÁLISIS:
• Variables analizadas (personales + laborales): 15
• Variables personales significativas: 2
• Variables laborales significativas: 2
• Variables binarias significativas: 2
• Variables temporales (excluidas): 4
• Ruta de resultados: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3

📈 DISTRIBUCIÓN DE CATEGORÍAS DE AUSENTISMO:
  • Medio: 426 empleados (52.9%)
  • Bajo: 306 empleados (38.0%)
  • Alto: 74 empleados (9.2%)

🏆 TOP 5 VARIABLES MÁS IMPORTANTES (PERSONALES Y LABORALES):
1. Disciplinary_failure (LABORAL):
   - ρ = -0.390 (Fuerza_Spearman    Moderada
Fuerza_Spearman    Moderada
Name: 14, dtype: object)
   - Cohen's d = -0.259 (Pequeño)
   - Disminuye ausentismo
2. Transportation_expense (LABORAL):
   - ρ = 0.170 (Fuerza_Spearman    Débil
Fuerza_Spearman    Débil
Name: 9, dtype: object)
   - Cohen's d = 0.032 (Muy peq

10. Reporte Final y Métricas

In [38]:
print("\n" + "="*80)
print("ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO")
print("="*80)

# Métricas finales
vars_analizadas = len(VARIABLES_NUMERICAS) + len(VARIABLES_CATEGORICAS)
vars_significativas_numeric = len(df_results_numeric[df_results_numeric['Significativa_Spearman']])
vars_significativas_categorical = df_results_categorical['Significativa_Principal'].sum()
vars_significativas_binarias = len(df_results_binarias_corregido[df_results_binarias_corregido['Significativa']]) if 'df_results_binarias_corregido' in locals() else 0

# Contar variables personales y laborales significativas
vars_personales_significativas = [var for var in VARIABLES_PERSONALES 
                                 if var in df_results_numeric['Variable'].values 
                                 and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

vars_laborales_significativas = [var for var in VARIABLES_LABORALES 
                                if var in df_results_numeric['Variable'].values 
                                and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

print(f"📊 MÉTRICAS DEL ANÁLISIS:")
print(f"• Variables analizadas: {vars_analizadas}")
print(f"• Variables numéricas significativas: {vars_significativas_numeric}")
print(f"• Variables categóricas significativas: {vars_significativas_categorical}")
print(f"• Variables binarias significativas: {vars_significativas_binarias}")
print(f"• Variables PERSONALES significativas: {len(vars_personales_significativas)}")
print(f"• Variables LABORALES significativas: {len(vars_laborales_significativas)}")
print(f"• Ruta de resultados: {config.OUTPUT_PATH}")

# Distribución de categorías de ausentismo
distribucion_ausentismo = df_clean[TARGET_CATEGORICO].value_counts()
print(f"\n📈 DISTRIBUCIÓN DE CATEGORÍAS DE AUSENTISMO:")
for categoria, count in distribucion_ausentismo.items():
    porcentaje = (count / len(df_clean)) * 100
    print(f"  • {categoria}: {count} empleados ({porcentaje:.1f}%)")

# Variables más importantes (solo significativas)
if not df_results_numeric.empty:
    variables_significativas = df_results_numeric[df_results_numeric['Significativa_Spearman']]
    if not variables_significativas.empty:
        top_variables = variables_significativas.nlargest(5, 'abs_corr')
        print(f"\n🏆 TOP 5 VARIABLES MÁS IMPORTANTES (SIGNIFICATIVAS):")
        for i, (_, row) in enumerate(top_variables.iterrows(), 1):
            direccion = "Aumenta" if row['Correlacion_Spearman'] > 0 else "Disminuye"
            # Clasificar tipo
            tipo = "PERSONAL" if row['Variable'] in VARIABLES_PERSONALES else "LABORAL" if row['Variable'] in VARIABLES_LABORALES else "TEMPORAL"
            print(f"{i}. {row['Variable']} ({tipo}): ρ = {row['Correlacion_Spearman']:.3f} - {direccion} ausentismo")

# Interpretación contextual mejorada
def interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido):
    print(f"\n🔍 HALLAZGOS CLAVE POR TIPO DE VARIABLE:")
    
    print("\n  🎯 VARIABLES PERSONALES:")
    personales_claves = {
        'Son': "Empleados con hijos tienen mayor ausentismo",
        'Social_drinker': "Consumo social de alcohol asociado con más ausentismo",
        'Age': "La edad muestra relación con patrones de ausentismo",
        'Body_mass_index': "Índice de masa corporal relacionado con salud y ausentismo"
    }
    
    for var, interpret in personales_claves.items():
        if var in df_results_numeric['Variable'].values:
            resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
            if resultado['Significativa_Spearman']:
                print(f"    • {var}: {interpret} (ρ={resultado['Correlacion_Spearman']:.3f})")
    
    print("\n  💼 VARIABLES LABORALES:")
    laborales_claves = {
        'Transportation_expense': "Mayor gasto en transporte asociado con más ausentismo",
        'Disciplinary_failure': "Fallas disciplinarias relacionadas con patrones de ausentismo", 
        'Service_time': "Tiempo de servicio en la empresa afecta ausentismo",
        'Hit_target': "Cumplimiento de metas influye en ausentismo"
    }
    
    for var, interpret in laborales_claves.items():
        if var in df_results_numeric['Variable'].values:
            resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
            if resultado['Significativa_Spearman']:
                print(f"    • {var}: {interpret} (ρ={resultado['Correlacion_Spearman']:.3f})")
    
    # Hallazgos binarios CORREGIDOS
    if df_results_binarias_corregido is not None:
        print("\n  🔘 VARIABLES BINARIAS:")
        for _, row in df_results_binarias_corregido[df_results_binarias_corregido['Significativa']].iterrows():
            tipo = "PERSONAL" if row['Variable'] in ['Social_drinker', 'Social_smoker'] else "LABORAL"
            if row['Variable'] == 'Disciplinary_failure':
                print(f"    • {row['Variable']} ({tipo}): Empleados con fallas disciplinarias tienen MENOS ausentismo")
                print(f"      - Diferencia: {row['Media_Grupo1']:.1f} vs {row['Media_Grupo0']:.1f} horas")
            elif row['Variable'] == 'Social_drinker':
                print(f"    • {row['Variable']} ({tipo}): Bebedores sociales tienen MÁS ausentismo")
                print(f"      - Diferencia: {row['Media_Grupo1']:.1f} vs {row['Media_Grupo0']:.1f} horas")

interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido)

print(f"\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE!")
print(f"📁 Resultados guardados en: {config.OUTPUT_PATH}")
print(f"📄 Informe ejecutivo: {config.OUTPUT_PATH}/INFORME_EJECUTIVO_DETALLADO.txt")
print(f"🔮 Próximo paso: Usar variables significativas para regresiones lineales multivariables")


ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO
📊 MÉTRICAS DEL ANÁLISIS:
• Variables analizadas: 27
• Variables numéricas significativas: 4
• Variables categóricas significativas: 7
• Variables binarias significativas: 2
• Variables PERSONALES significativas: 2
• Variables LABORALES significativas: 2
• Ruta de resultados: G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results\Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_3

📈 DISTRIBUCIÓN DE CATEGORÍAS DE AUSENTISMO:
  • Medio: 426 empleados (52.9%)
  • Bajo: 306 empleados (38.0%)
  • Alto: 74 empleados (9.2%)

🏆 TOP 5 VARIABLES MÁS IMPORTANTES (SIGNIFICATIVAS):
1. Disciplinary_failure (LABORAL): ρ = -0.390 - Disminuye ausentismo
2. Transportation_expense (LABORAL): ρ = 0.170 - Aumenta ausentismo
3. Son (PERSONAL): ρ = 0.132 - Aumenta ausentismo
4. Social_drinker (PERSONAL): ρ = 0.121 - Aumenta ausentismo

🔍 HALLAZGOS CLAVE POR TIPO DE VARIABLE:

  🎯 VARIABLES PERSONALES:
    • Son: Empleados con